# Evaluación Final Transversal — Sistema multimodal de imágenes y texto

## Parte 3: Fusión de las representaciones de imagen y texto en un clasificador conjunto (PyTorch)

**Institución:** Duoc UC, Escuela de Informática y Telecomunicaciones<br>
**Asignatura:** Deep Learning (DLY0100) · Sección 003D<br>
**Docente:** Guillermo Eugenio Pinto Fuentes<br>
**Integrantes:** Esteban Pacheco Vega, Francisco Figueroa Donoso<br>
**Fecha:** 27-11-2026<br>
**Repositorio:** https://github.com/Fran-informatica/EFT_EPACHECO_FFIGUEROA_DLY0100_003D

## 1. Introducción

### 1.1 Descripción del problema

El sistema recibe **una imagen de una prenda y una reseña de texto** y debe entregar una predicción conjunta. Los dos primeros notebooks entrenaron por separado una red convolucional sobre Fashion-MNIST (10 prendas) y modelos de texto sobre reseñas de IMDB (sentimiento positivo o negativo), y guardaron sus modelos finales. Este notebook los integra:

1. Carga ambos modelos **congelados** (sus pesos no cambian) y extrae de cada uno su **representación**: el vector de 128 valores de su última capa oculta.
2. Forma pares (imagen, reseña). Como no existe un dataset real con imágenes de prendas y reseñas asociadas, cada imagen se empareja **al azar** con una reseña de su misma partición (entrenamiento, validación o test). El emparejamiento es sintético: la prenda no influye en el sentimiento ni al revés.
3. Entrena un **clasificador fusionado** que recibe ambas representaciones y predice una **clase conjunta de 20 categorías**: prenda × sentimiento (por ejemplo, «Bolso · positiva»).

Se comparan dos técnicas de fusión:

- **Concatenación + MLP:** une los dos vectores en uno de 256 valores y lo clasifica con un perceptrón multicapa.
- **Atención entre modalidades:** trata cada representación como un *token* y aplica un bloque de autoatención, en el que cada modalidad pondera cuánto toma de la otra antes de clasificar.

### 1.2 Objetivo del modelo

Entrenar y optimizar el clasificador fusionado, compararlo con lo que logra cada modalidad por separado y elegir con evidencia propia la técnica de fusión.

- Ninguna modalidad sola puede resolver la tarea: la imagen no dice nada del sentimiento y la reseña no dice nada de la prenda. El aporte de la fusión se mide frente a clasificadores que reciben solo una de las dos representaciones.
- **Métrica de decisión: F1 macro en validación** sobre las 20 clases conjuntas, la misma métrica de los otros dos notebooks.
- Como el emparejamiento es aleatorio, prenda y sentimiento son independientes: el mejor resultado posible se acerca a combinar las predicciones de ambos modelos por separado. Esa **fusión tardía** (sección 7) es la referencia más exigente.

**Nota de terminología:** *accuracy* = **exactitud**; *precision* = **precisión**; *recall* = **exhaustividad**; F1 = media armónica de precisión y exhaustividad. El promedio **macro** pondera igual a las 20 clases y el **ponderado** según su cantidad de ejemplos.

### 1.3 Estructura del notebook

1. Introducción.
2. Configuración inicial, reutilización de resultados y herramientas.
3. Carga de los modelos entrenados de imagen y de texto.
4. Datos de ambas modalidades con las mismas particiones de los notebooks anteriores.
5. Extracción de las representaciones congeladas.
6. Emparejamiento sintético y clase conjunta.
7. Líneas base: clase mayoritaria y fusión tardía.
8. Utilidades de entrenamiento, evaluación y experimentos.
9. Arquitecturas de fusión y configuración base.
10. Modelo base y rango de error.
11. Experimento clave: aporte de cada modalidad.
12. Experimento clave: técnica de fusión.
13. a 15. Experimentos de preprocesamiento, entrenamiento, arquitectura y regularización.
16. Modelo final.
17. Evaluación en test.
18. Comparación entre la configuración base y la final.
19. Síntesis de los experimentos.
20. Verificaciones de la dependencia del orden.
21. Pesos de la atención entre modalidades.
22. Demostración del sistema completo.
23. a 25. Conclusiones, conclusión final y referencias.
26. a 30. Anexos (gráficos para la presentación y herramientas), respaldo, descripción de las salidas y cierre del entorno de ejecución.

**Criterio metodológico:**

- Todas las decisiones se toman con los pares de **validación**. Los pares de **test** se usan una sola vez, con el modelo final.
- Se cambia **un hiperparámetro a la vez**; la variante elegida se incorpora y el experimento siguiente parte de ella.
- Antes de comparar se mide el **rango de error** entre reentrenamientos con 5 semillas. Una diferencia menor o igual a ese rango es un **empate técnico**, y los empates se deciden según el contexto del problema, no solo por la métrica.
- Si la mejor variante queda en el borde del rango probado, el rango se amplía un paso en esa dirección.
- Al final se verifican las combinaciones con interacción conocida y se compara el modelo final con el base usando las mismas semillas.

## 2. Configuración inicial

Se importan en un único bloque todas las librerías del notebook:

- **PyTorch** (`torch.nn`, `pack_padded_sequence`): reconstruye la CNN y el modelo de texto, y define y entrena los clasificadores fusionados.
- **NumPy / Pandas**: arreglos y tablas comparativas.
- **scikit-learn**: métricas (exactitud, precisión, exhaustividad, F1 y matriz de confusión).
- **Plotly**: gráficos interactivos. El renderizador `vscode+colab` permite verlos tanto en Colab como en VS Code conectado a un servidor de Colab.
- **IPython.display**: tablas con `display`.
- **re / gzip / io / urllib / os / json / shutil**: tokenización, lectura de los datos y de los modelos desde el repositorio y respaldo.
- **copy / math / random / time**: copia de los pesos de la mejor época, escalas de la atención, semillas y tiempos.

`fijar_semilla(semilla)` fija las semillas de `random`, NumPy y PyTorch (CPU y GPU). No retorna nada.

In [ ]:
# ============================================================
# Importación de librerías (todos los imports del notebook van aquí)
# ============================================================
import copy
import gzip
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from torch.nn.utils.rnn import pack_padded_sequence
try:
    from google.colab import files, runtime
except ImportError:
    files = runtime = None

# ============================================================
# Reproducibilidad, dispositivo y visualización
# ============================================================
SEED = 42


def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


fijar_semilla(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pio.renderers.default = "vscode+colab"
COLORES = ["#104281", "#3987E5", "#86B6EF"]
AMARILLO = "#F7B500"
print("PyTorch", torch.__version__, "| dispositivo:", device)

**Nota:** aun con semilla fija, el entrenamiento en GPU no es completamente determinista. Por eso, antes de comparar configuraciones se mide ese margen de variación reentrenando la configuración base con varias semillas.

**Requisito de ejecución:** activar GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*). La fusión entrena sobre vectores ya extraídos y es rápida; la GPU acelera sobre todo la extracción de las representaciones de texto.

### 2.1 Reutilización de resultados ya obtenidos

Las métricas de cada corrida se guardan en `resultados_fusion_guardados.json` y se reutilizan cuando la configuración y la semilla coinciden **exactamente** (la clave incluye todos los hiperparámetros, la semilla, el tope de épocas y la paciencia). Cada corrida nueva se imprime como una línea `CORRIDA_GUARDADA {...}`, para reconstruir el JSON desde el notebook guardado si la sesión se corta, sin montar Google Drive.

- Las corridas que necesitan el modelo entrenado o sus curvas (modelo base, modalidades, técnica de fusión, épocas y modelo final) no se pueden reutilizar solo con las métricas: además se guardan como `.pt`.
- `leer_archivo(ruta, binario)` busca el archivo en la carpeta de trabajo y, si no está, lo descarga de `URL_REPO` (la dirección *raw* del repositorio del grupo). Retorna el contenido, o `None` si no está en ninguno de los dos lugares.
- `guardar_corridas()` escribe en el JSON las corridas guardadas más las nuevas.
- `guardar_modelo_corrida(clave, modelo, historia, fila)` guarda esas corridas en `corridas_fusion/corrida_fusion_<código>.pt`, con los pesos del modelo, la historia por época y la fila de métricas; el código es un *hash* de la clave. `cargar_modelo_corrida(clave, modelo)` busca ese archivo en `corridas_fusion/` y en la raíz, primero en la carpeta de trabajo y después en el repositorio, y si lo encuentra carga los pesos en `modelo` (con `weights_only=True`) y retorna `(fila, historia)`; si no, retorna `None`. Así, al volver a ejecutar, esas corridas tampoco se entrenan. Con `REUTILIZAR_MODELOS = False` se entrenan siempre.
- `descargar_resultados()` descarga al equipo el JSON y los `.pt` nuevos cada vez que termina una celda de código que agregó corridas, para no perderlos si la sesión de Colab se cierra. Se activa con `DESCARGAR_RESULTADOS = True`. El navegador pide permiso para descargas múltiples la primera vez y numera las copias repetidas del JSON (`(1)`, `(2)`, …): la vigente es la más reciente, porque cada una contiene todas las corridas anteriores. Para reutilizarlos, el JSON (sin el número) va en la raíz del repositorio y los `.pt` en `corridas_fusion/` o en la raíz. Fuera de Colab (por ejemplo, en VS Code con kernel de Colab) la descarga puede no funcionar: en ese caso se imprime un aviso y los archivos quedan en la carpeta de trabajo.

In [ ]:
# ============================================================
# Carga de los resultados guardados
# ============================================================
REUTILIZAR_RESULTADOS = True
ARCHIVO_RESULTADOS = "resultados_fusion_guardados.json"

URL_REPO = "https://raw.githubusercontent.com/Fran-informatica/EFT_EPACHECO_FFIGUEROA_DLY0100_003D/main/"


def leer_archivo(ruta, binario=False):
    if os.path.exists(ruta):
        with open(ruta, "rb") as f:
            datos = f.read()
    else:
        try:
            with urllib.request.urlopen(URL_REPO + urllib.parse.quote(ruta)) as r:
                datos = r.read()
        except urllib.error.HTTPError:
            return None
    return datos if binario else datos.decode("utf-8")


def guardar_corridas():
    with open(ARCHIVO_RESULTADOS, "w", encoding="utf-8") as f:
        json.dump({**RESULTADOS_GUARDADOS, **CORRIDAS_NUEVAS}, f, ensure_ascii=False, indent=1, default=float)


texto_resultados = leer_archivo(ARCHIVO_RESULTADOS) if REUTILIZAR_RESULTADOS else None
RESULTADOS_GUARDADOS = json.loads(texto_resultados) if texto_resultados else {}
CORRIDAS_NUEVAS = {}
REUTILIZAR_MODELOS = True
CARPETA_MODELOS = "corridas_fusion"
DESCARGAR_RESULTADOS = True
MODELOS_NUEVOS = []
descargados = {"corridas": 0, "modelos": 0}


def nombre_modelo_corrida(clave):
    return f"corrida_fusion_{hashlib.sha1(clave.encode('utf-8')).hexdigest()[:16]}.pt"


def guardar_modelo_corrida(clave, modelo, historia, fila):
    os.makedirs(CARPETA_MODELOS, exist_ok=True)
    ruta = f"{CARPETA_MODELOS}/{nombre_modelo_corrida(clave)}"
    metadatos = json.dumps({"clave": clave, "fila": fila, "historia": historia}, default=float)
    torch.save({"estado": modelo.state_dict(), "metadatos": metadatos}, ruta)
    MODELOS_NUEVOS.append(ruta)


def cargar_modelo_corrida(clave, modelo):
    nombre = nombre_modelo_corrida(clave)
    for ruta in [f"{CARPETA_MODELOS}/{nombre}", nombre]:
        datos = leer_archivo(ruta, binario=True)
        if datos is None:
            continue
        guardado = torch.load(io.BytesIO(datos), map_location=device, weights_only=True)
        metadatos = json.loads(guardado["metadatos"])
        if metadatos["clave"] != clave:
            continue
        modelo.load_state_dict(guardado["estado"])
        return metadatos["fila"], metadatos["historia"]
    return None


def descargar_resultados(resultado=None):
    if not DESCARGAR_RESULTADOS or len(CORRIDAS_NUEVAS) == descargados["corridas"] and len(MODELOS_NUEVOS) == descargados["modelos"]:
        return
    pendientes = [ARCHIVO_RESULTADOS] + MODELOS_NUEVOS[descargados["modelos"]:]
    descargados["corridas"], descargados["modelos"] = len(CORRIDAS_NUEVAS), len(MODELOS_NUEVOS)
    try:
        for ruta in pendientes:
            files.download(ruta)
    except Exception as error:
        print(f"Descarga automática no disponible ({type(error).__name__}): los archivos quedan en la carpeta de trabajo")


eventos = get_ipython().events
for funcion in [f for f in eventos.callbacks["post_run_cell"] if getattr(f, "__name__", "") in ("descargar_json", "descargar_resultados")]:
    eventos.unregister("post_run_cell", funcion)
eventos.register("post_run_cell", descargar_resultados)
print(f"Corridas guardadas disponibles: {len(RESULTADOS_GUARDADOS)}")

### 2.2 Herramientas: fortalezas y debilidades

| Herramienta | Fortaleza en este proyecto | Debilidad o costo |
|---|---|---|
| **PyTorch** | Las tres redes están en el mismo framework: los modelos guardados se reconstruyen y se combinan sin conversiones, y la fusión reutiliza el bloque de atención del notebook de texto | Hay que repetir aquí la definición de las clases de cada modelo para poder cargar sus pesos |
| **Representaciones congeladas** | Se extraen una sola vez; cada corrida de fusión entrena solo un clasificador pequeño, en segundos | Los modelos de imagen y de texto no se ajustan a la tarea conjunta (se discute en las mejoras) |
| **Modelos guardados con `weights_only=True`** | La carga solo acepta tensores y tipos básicos: no ejecuta código serializado de un archivo descargado | Obliga a guardar los índices como tensores y la configuración con tipos simples |
| **Google Colab con GPU T4** | Extrae las representaciones de las 70.000 imágenes y las 49.582 reseñas en poco tiempo | Cuota de uso limitada; los archivos quedan en el servidor y hay que descargarlos |
| **Plotly** | Gráficos interactivos y exportables a HTML y PNG para la presentación | Los gráficos pesan más en el notebook que una imagen estática |

## 3. Carga de los modelos entrenados

### 3.1 Archivos de los modelos

Los notebooks anteriores guardan `cnn_final.pt` (en `respaldo_eft_cnn/`) y `texto_final.pt`. Para esta ejecución deben estar en la carpeta de trabajo o en la raíz del repositorio. Cada archivo contiene la configuración del modelo, sus pesos y los índices de la partición con que se entrenó; el de la CNN además trae los estadísticos de normalización y el de texto, el vocabulario.

`cargar_modelo(nombre, alternativas)` busca el archivo en la carpeta de trabajo (con su nombre o en las rutas alternativas del respaldo) y, si no está, en el repositorio. Lo carga con `weights_only=True`, que solo acepta tensores y tipos básicos. Retorna el diccionario guardado.

In [ ]:
# ============================================================
# Lectura de los archivos de los modelos
# ============================================================
def cargar_modelo(nombre, alternativas=()):
    ruta = next((r for r in alternativas if os.path.exists(r)), nombre)
    datos = leer_archivo(ruta, binario=True)
    if datos is None:
        raise FileNotFoundError(f"No se encontró {nombre}: subirlo a la carpeta de trabajo o a la raíz del repositorio")
    return torch.load(io.BytesIO(datos), map_location=device, weights_only=True)


guardado_cnn = cargar_modelo("cnn_final.pt", ["respaldo_eft_cnn/cnn_final.pt"])
guardado_texto = cargar_modelo("texto_final.pt", ["respaldo_eft_texto/texto_final.pt"])
print("CNN:", guardado_cnn["config"])
print("Texto:", guardado_texto["familia"], guardado_texto["config"])

### 3.2 Arquitecturas de los modelos de imagen y de texto

Para cargar los pesos se necesitan las mismas clases con que se entrenaron. Se repiten aquí, sin cambios, las definiciones de los notebooks anteriores: `ClasificadorCNN` (bloques convolucionales y capa densa de representación), los bloques de atención (`MultiHeadAttentionLayer`, `EncoderBlock`, `DecoderLayer`) y los clasificadores de texto (`ClasificadorRecurrente` y `ClasificadorTransformer`). Su funcionamiento se explica en los notebooks de imagen y de texto. Todos ofrecen el método `representacion()`, que retorna el vector de 128 valores de la última capa oculta.

In [ ]:
# ============================================================
# Definiciones de la CNN y de los modelos de texto
# ============================================================
ACTIVACIONES = {"relu": nn.ReLU, "leaky_relu": lambda: nn.LeakyReLU(0.01), "gelu": nn.GELU,
                "tanh": nn.Tanh, "sigmoide": nn.Sigmoid}


class ClasificadorCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        capas, canales, lado = [], 1, 28
        for i in range(config["bloques"]):
            salida = config["filtros"] * 2 ** i
            capas.append(nn.Conv2d(canales, salida, config["kernel"], padding=config["kernel"] // 2))
            if config["batchnorm"]:
                capas.append(nn.BatchNorm2d(salida))
            capas += [ACTIVACIONES[config["activacion"]](), nn.MaxPool2d(2)]
            canales, lado = salida, lado // 2
        self.extractor = nn.Sequential(*capas, nn.Flatten())
        self.densa = nn.Sequential(nn.Dropout(config["dropout"]),
                                   nn.Linear(canales * lado * lado, config["unidades_densas"]),
                                   ACTIVACIONES[config["activacion"]]())
        self.salida = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(config["unidades_densas"], 10))

    def representacion(self, x):
        return self.densa(self.extractor(x))

    def forward(self, x):
        return self.salida(self.representacion(x))


class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout):
        super().__init__()
        assert hid_dim % n_heads == 0
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        self.head_dim = hid_dim // n_heads
        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)
        self.fc_o = nn.Linear(hid_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        # Q = [batch size, query len, hid dim] -> [batch size, n heads, query len, head dim]
        Q = self.fc_q(query).view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = self.fc_k(key).view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = self.fc_v(value).view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        # energy = [batch size, n heads, query len, key len]
        if mask is not None:
            energy = energy.masked_fill(~mask, -1e10)
        attention = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attention), V)
        x = x.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.hid_dim)
        # x = [batch size, query len, hid dim]
        return self.fc_o(x), attention


class PositionwiseFeedforwardLayer(nn.Module):
    def __init__(self, hid_dim, pf_dim, dropout):
        super().__init__()
        self.fc_1 = nn.Linear(hid_dim, pf_dim)
        self.fc_2 = nn.Linear(pf_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc_2(self.dropout(torch.relu(self.fc_1(x))))


class EncoderBlock(nn.Module):
    def __init__(self, hid_dim, n_heads, pf_dim, dropout):
        super().__init__()
        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)
        self.guardar_atencion, self.atencion = False, None

    def forward(self, src, src_mask):
        # src = [batch size, src len, hid dim]; src_mask = [batch size, 1, 1, src len]
        _src, atencion = self.self_attention(src, src, src, src_mask)
        if self.guardar_atencion:
            self.atencion = atencion.detach()
        src = self.self_attn_layer_norm(src + self.dropout(_src))
        _src = self.positionwise_feedforward(src)
        return self.ff_layer_norm(src + self.dropout(_src))


class DecoderLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, pf_dim, dropout):
        super().__init__()
        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.enc_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout)
        self.encoder_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)
        self.guardar_atencion, self.atencion = False, None

    def forward(self, trg, enc_src, trg_mask, src_mask):
        # trg = [batch size, trg len, hid dim]; enc_src = [batch size, src len, hid dim]
        _trg, _ = self.self_attention(trg, trg, trg, trg_mask)
        trg = self.self_attn_layer_norm(trg + self.dropout(_trg))
        _trg, atencion = self.encoder_attention(trg, enc_src, enc_src, src_mask)
        if self.guardar_atencion:
            self.atencion = atencion.detach()
        trg = self.enc_attn_layer_norm(trg + self.dropout(_trg))
        _trg = self.positionwise_feedforward(trg)
        return self.ff_layer_norm(trg + self.dropout(_trg))


CELDAS = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}
UNIDADES_REPRESENTACION = 128


class ClasificadorRecurrente(nn.Module):
    def __init__(self, config, n_vocab, pad_idx):
        super().__init__()
        self.bidireccional = config["bidireccional"]
        self.embedding = nn.Embedding(n_vocab, config["embed_dim"], padding_idx=pad_idx)
        self.rnn = CELDAS[config["celda"]](config["embed_dim"], config["units"], num_layers=config["capas"],
                                           batch_first=True, bidirectional=config["bidireccional"],
                                           dropout=config["dropout"] if config["capas"] > 1 else 0.0)
        salida_rnn = config["units"] * (2 if self.bidireccional else 1)
        self.densa = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(salida_rnn, UNIDADES_REPRESENTACION), nn.ReLU())
        self.salida = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(UNIDADES_REPRESENTACION, 2))

    def representacion(self, x, largos):
        # x = [batch size, seq len]
        empaquetado = pack_padded_sequence(self.embedding(x), largos.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(empaquetado)
        if isinstance(h, tuple):
            h = h[0]  # la LSTM devuelve (h, c)
        # h = [capas * direcciones, batch size, units]
        final = torch.cat([h[-2], h[-1]], dim=1) if self.bidireccional else h[-1]
        return self.densa(final)

    def forward(self, x, largos):
        return self.salida(self.representacion(x, largos))


class ClasificadorTransformer(nn.Module):
    def __init__(self, config, n_vocab, pad_idx, cls_idx):
        super().__init__()
        hid = config["hid_dim"]
        self.agregacion, self.pad_idx, self.cls_idx = config["agregacion"], pad_idx, cls_idx
        self.tok_embedding = nn.Embedding(n_vocab, config["embed_dim"])
        self.proyeccion = nn.Linear(config["embed_dim"], hid) if config["embed_dim"] != hid else nn.Identity()
        self.pos_embedding = nn.Embedding(config["maxlen"] + 1, hid)
        self.bloques = nn.ModuleList([EncoderBlock(hid, config["cabezas"], config["pf_dim"], config["dropout"])
                                      for _ in range(config["capas"])])
        if self.agregacion == "decoder":
            self.consulta = nn.Parameter(torch.randn(1, 1, hid) * 0.02)
            self.decoder = DecoderLayer(hid, config["cabezas"], config["pf_dim"], config["dropout"])
        self.dropout = nn.Dropout(config["dropout"])
        self.escala = math.sqrt(hid)
        self.densa = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(hid, UNIDADES_REPRESENTACION), nn.ReLU())
        self.salida = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(UNIDADES_REPRESENTACION, 2))

    def codificar(self, x):
        if self.agregacion == "cls":
            x = torch.cat([torch.full((x.shape[0], 1), self.cls_idx, device=x.device), x], dim=1)
        mascara = (x != self.pad_idx)[:, None, None, :]
        # mascara = [batch size, 1, 1, src len]
        pos = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        h = self.dropout(self.proyeccion(self.tok_embedding(x)) * self.escala + self.pos_embedding(pos))
        for bloque in self.bloques:
            h = bloque(h, mascara)
        # h = [batch size, src len, hid dim]
        return h, mascara

    def representacion(self, x, largos):
        h, mascara = self.codificar(x)
        if self.agregacion == "promedio":
            validos = mascara[:, 0, 0, :].unsqueeze(-1).float()
            vector = (h * validos).sum(1) / validos.sum(1)
        elif self.agregacion == "cls":
            vector = h[:, 0]
        else:
            consulta = self.consulta.expand(x.shape[0], -1, -1)
            vector = self.decoder(consulta, h, None, mascara)[:, 0]
        return self.densa(vector)

    def forward(self, x, largos):
        return self.salida(self.representacion(x, largos))


def inicializar_pesos(m):
    if isinstance(m, (nn.Linear, nn.Embedding)):
        nn.init.xavier_uniform_(m.weight.data)


def construir_modelo(config, n_vocab, pad_idx, cls_idx):
    if config["familia"] == "recurrente":
        return ClasificadorRecurrente(config, n_vocab, pad_idx).to(device)
    modelo = ClasificadorTransformer(config, n_vocab, pad_idx, cls_idx)
    modelo.apply(inicializar_pesos)
    return modelo.to(device)

### 3.3 Reconstrucción y congelamiento

Se reconstruye cada modelo con su configuración, se cargan sus pesos, se pasa a modo evaluación (sin Dropout) y se desactivan sus gradientes: los pesos quedan **congelados** y solo se entrena el clasificador fusionado.

In [ ]:
# ============================================================
# Reconstrucción de los modelos congelados
# ============================================================
ITOS = guardado_texto["itos"]
STOI = {t: i for i, t in enumerate(ITOS)}
PAD_IDX, CLS_IDX, UNK_IDX = STOI["<pad>"], STOI["<cls>"], STOI["<unk>"]
CONFIG_CNN, CONFIG_TEXTO = guardado_cnn["config"], guardado_texto["config"]
MAXLEN = CONFIG_TEXTO["maxlen"]

modelo_cnn = ClasificadorCNN(CONFIG_CNN).to(device)
modelo_cnn.load_state_dict(guardado_cnn["estado"])
modelo_texto = construir_modelo(CONFIG_TEXTO, len(ITOS), PAD_IDX, CLS_IDX)
modelo_texto.load_state_dict(guardado_texto["estado"])
for modelo in (modelo_cnn, modelo_texto):
    modelo.eval()
    for p in modelo.parameters():
        p.requires_grad_(False)
print(f"CNN: {sum(p.numel() for p in modelo_cnn.parameters()):,} parámetros congelados")
print(f"Texto ({guardado_texto['familia']}): {sum(p.numel() for p in modelo_texto.parameters()):,} parámetros congelados")

## 4. Datos de ambas modalidades

### 4.1 Imágenes de Fashion-MNIST

Se leen los mismos archivos IDX del notebook de la CNN y se reconstruye su partición con los índices guardados en `cnn_final.pt`: 50.000 imágenes de entrenamiento, 10.000 de validación y las 10.000 de test. Usar la misma partición garantiza que ninguna imagen de validación o de test haya influido en el entrenamiento de la CNN.

`leer_idx(nombre)` lee un archivo IDX (y lo descomprime si viene en gzip) y retorna un arreglo de NumPy con la forma de su cabecera.

In [ ]:
# ============================================================
# Lectura de Fashion-MNIST y partición de la CNN
# ============================================================
CLASES = guardado_cnn["clases"]


def leer_idx(nombre):
    datos = leer_archivo(nombre, binario=True)
    if datos is None:
        raise FileNotFoundError(f"No se encontró {nombre} ni en la carpeta de trabajo ni en el repositorio")
    if datos[:2] == b"\x1f\x8b":
        datos = gzip.decompress(datos)
    dimensiones = datos[3]
    forma = [int.from_bytes(datos[4 + 4 * i:8 + 4 * i], "big") for i in range(dimensiones)]
    return np.frombuffer(datos, dtype=np.uint8, offset=4 + 4 * dimensiones).reshape(forma)


x_completo, y_completo = leer_idx("train-images-idx3-ubyte"), leer_idx("train-labels-idx1-ubyte").astype(np.int64)
IMAGENES = {"train": x_completo[guardado_cnn["idx_train"].cpu().numpy()],
            "val": x_completo[guardado_cnn["idx_val"].cpu().numpy()],
            "test": np.array(leer_idx("t10k-images-idx3-ubyte"))}
Y_PRENDA = {"train": y_completo[guardado_cnn["idx_train"].cpu().numpy()],
            "val": y_completo[guardado_cnn["idx_val"].cpu().numpy()],
            "test": leer_idx("t10k-labels-idx1-ubyte").astype(np.int64)}
display(pd.DataFrame({p: {"imágenes": len(Y_PRENDA[p])} for p in IMAGENES}).T)

### 4.2 Reseñas de IMDB

Se lee el mismo CSV del notebook de texto, se eliminan los duplicados de la misma forma (se conserva la primera copia de cada texto) y se reconstruye la partición con los índices guardados en `texto_final.pt`. Cada reseña se limpia, se tokeniza con la misma función `tokenizar` y se codifica con el vocabulario guardado; las desconocidas van a `<unk>`.

`codificar(listas_tokens, y)` retorna un diccionario con la matriz de índices (en GPU), los largos y las etiquetas, con el mismo formato del notebook de texto.

In [ ]:
# ============================================================
# Lectura, limpieza y codificación de las reseñas
# ============================================================
PATRON_HTML = re.compile(r"<br\s*/?>")
PATRON_TOKENS = re.compile(r"[a-z0-9']+|[!?.,;:]")


def tokenizar(texto):
    return PATRON_TOKENS.findall(PATRON_HTML.sub(" ", texto.lower()))

SENTIMIENTOS = ["negativa", "positiva"]
texto_csv = leer_archivo("IMDB Dataset.csv")
if texto_csv is None:
    raise FileNotFoundError("No se encontró IMDB Dataset.csv ni en la carpeta de trabajo ni en el repositorio")
imdb = pd.read_csv(io.StringIO(texto_csv)).drop_duplicates("review").reset_index(drop=True)
imdb["y"] = (imdb["sentiment"] == "positive").astype(np.int64)


def codificar(listas_tokens, y):
    x = torch.full((len(listas_tokens), MAXLEN), PAD_IDX, dtype=torch.long)
    largos = torch.zeros(len(listas_tokens), dtype=torch.long)
    for fila, toks in enumerate(listas_tokens):
        ids = [STOI.get(t, UNK_IDX) for t in toks[:MAXLEN]] or [UNK_IDX]
        x[fila, :len(ids)] = torch.tensor(ids)
        largos[fila] = len(ids)
    return {"x": x.to(device), "largos": largos.to(device), "largos_cpu": largos, "y": torch.tensor(y, device=device)}


RESENAS, Y_SENTIMIENTO, TEXTOS = {}, {}, {}
for p in ["train", "val", "test"]:
    indices = guardado_texto[f"idx_{p}"].cpu().numpy()
    TEXTOS[p] = imdb["review"].to_numpy()[indices]
    Y_SENTIMIENTO[p] = imdb["y"].to_numpy()[indices]
    RESENAS[p] = codificar([tokenizar(t) for t in TEXTOS[p]], Y_SENTIMIENTO[p])
display(pd.DataFrame({p: {"reseñas": len(Y_SENTIMIENTO[p]), "% positivas": Y_SENTIMIENTO[p].mean() * 100}
                      for p in RESENAS}).T.round(2))

### 4.3 Verificación de los modelos cargados

Antes de fusionar se comprueba que cada modelo reconstruido rinde lo mismo que en su notebook: se calcula el F1 macro de validación de la CNN sobre las imágenes y del modelo de texto sobre las reseñas. Si las cifras no coinciden con las de los notebooks anteriores, el archivo cargado o el preprocesamiento no corresponden al modelo entrenado.

`preparar_imagenes(x, config, media, std)` convierte un lote de imágenes en enteros de 8 bits al formato de entrada de la CNN, con la normalización de su configuración. `predecir_rama(...)` retorna las clases predichas por la cabeza de clasificación original de cada modelo.

In [ ]:
# ============================================================
# Preparación de imágenes y predicción de cada rama
# ============================================================
MEDIA_PIXEL, STD_PIXEL = guardado_cnn["media_pixel"], guardado_cnn["std_pixel"]


def preparar_imagenes(x, config, media, std):
    x = torch.as_tensor(x, device=device).float().unsqueeze(1)
    if config["normalizacion"] == "escala_0_1":
        return x / 255
    if config["normalizacion"] == "estandar":
        return (x / 255 - media) / std
    return x


def lotes_ordenados(datos, lote):
    orden = torch.argsort(datos["largos_cpu"], descending=True).to(device)
    for inicio in range(0, len(orden), lote):
        indices = orden[inicio:inicio + lote]
        largos = datos["largos"][indices]
        yield indices, datos["x"][indices, :int(largos.max())], largos


def predecir_rama(modalidad, datos, lote=1024):
    with torch.no_grad():
        if modalidad == "imagen":
            return np.concatenate([modelo_cnn(preparar_imagenes(datos[i:i + lote], CONFIG_CNN, MEDIA_PIXEL, STD_PIXEL)).argmax(1).cpu().numpy()
                                   for i in range(0, len(datos), lote)])
        salida = torch.zeros(len(datos["largos_cpu"]), dtype=torch.long, device=device)
        for indices, x, largos in lotes_ordenados(datos, lote):
            salida[indices] = modelo_texto(x, largos).argmax(1)
        return salida.cpu().numpy()


PRED_PRENDA = {p: predecir_rama("imagen", IMAGENES[p]) for p in IMAGENES}
PRED_SENTIMIENTO = {p: predecir_rama("texto", RESENAS[p]) for p in RESENAS}
display(pd.DataFrame({
    "CNN (10 prendas)": {"f1_macro_val": f1_score(Y_PRENDA["val"], PRED_PRENDA["val"], average="macro"),
                          "exactitud_val": accuracy_score(Y_PRENDA["val"], PRED_PRENDA["val"])},
    "Texto (sentimiento)": {"f1_macro_val": f1_score(Y_SENTIMIENTO["val"], PRED_SENTIMIENTO["val"], average="macro"),
                            "exactitud_val": accuracy_score(Y_SENTIMIENTO["val"], PRED_SENTIMIENTO["val"])},
}).T.round(4))

**Observación:** *(completar después de ejecutar)*

- ¿El F1 macro de validación de cada rama coincide con el de su notebook?

## 5. Extracción de las representaciones congeladas

Cada imagen y cada reseña se pasa una sola vez por su modelo congelado y se guarda su vector de 128 valores (`representacion()`). Los clasificadores fusionados se entrenan después sobre estos vectores, de modo que cada corrida de la fusión no vuelve a procesar píxeles ni tokens.

`extraer_imagenes(x)` y `extraer_resenas(datos)` retornan tensores en GPU de forma `[ejemplos, 128]`, en el orden original de cada partición.

In [ ]:
# ============================================================
# Representaciones de imagen y de texto
# ============================================================
def extraer_imagenes(x, lote=2048):
    with torch.no_grad():
        return torch.cat([modelo_cnn.representacion(preparar_imagenes(x[i:i + lote], CONFIG_CNN, MEDIA_PIXEL, STD_PIXEL))
                          for i in range(0, len(x), lote)])


def extraer_resenas(datos, lote=512):
    salida = torch.zeros(len(datos["largos_cpu"]), UNIDADES_REPRESENTACION, device=device)
    with torch.no_grad():
        for indices, x, largos in lotes_ordenados(datos, lote):
            salida[indices] = modelo_texto.representacion(x, largos)
    return salida


inicio = time.time()
REP_IMAGEN = {p: extraer_imagenes(IMAGENES[p]) for p in IMAGENES}
REP_TEXTO = {p: extraer_resenas(RESENAS[p]) for p in RESENAS}
print(f"Extracción completa en {time.time() - inicio:.1f} s")
display(pd.DataFrame({p: {"imagen": tuple(REP_IMAGEN[p].shape), "texto": tuple(REP_TEXTO[p].shape)} for p in IMAGENES}).T)

## 6. Emparejamiento sintético y clase conjunta

### 6.1 Formación de los pares

Cada imagen de una partición se empareja con una reseña elegida al azar **de la misma partición**, con reposición y con la semilla fija. Así, ninguna reseña de test acompaña a una imagen de entrenamiento y viceversa. Como hay más imágenes que reseñas en cada partición (50.000 frente a 34.708 en entrenamiento y 10.000 frente a 7.437 en validación y test), algunas reseñas se usan más de una vez, cada vez con una imagen distinta.

La **clase conjunta** es `prenda × 2 + sentimiento`: 20 clases, de «Camiseta / top · negativa» (0) a «Botín · positiva» (19).

`emparejar(n_imagenes, n_resenas, semilla)` retorna, para cada imagen, el índice de la reseña que la acompaña.

In [ ]:
# ============================================================
# Emparejamiento imagen–reseña y clase conjunta
# ============================================================
CLASES_CONJUNTAS = [f"{prenda} · {sentimiento}" for prenda in CLASES for sentimiento in SENTIMIENTOS]


def emparejar(n_imagenes, n_resenas, semilla):
    return np.random.default_rng(semilla).integers(0, n_resenas, n_imagenes)


PARES = {p: emparejar(len(Y_PRENDA[p]), len(Y_SENTIMIENTO[p]), SEED + i) for i, p in enumerate(["train", "val", "test"])}
Y_CONJUNTA = {p: Y_PRENDA[p] * 2 + Y_SENTIMIENTO[p][PARES[p]] for p in PARES}
usos = pd.DataFrame({p: {"pares": len(PARES[p]), "reseñas disponibles": len(Y_SENTIMIENTO[p]),
                         "reseñas usadas al menos una vez": len(np.unique(PARES[p])),
                         "usos medios por reseña usada": len(PARES[p]) / len(np.unique(PARES[p])),
                         "usos máximos de una reseña": np.bincount(PARES[p]).max()} for p in PARES}).T
display(usos.round(2))

### 6.2 Distribución de las clases conjuntas

Con 10 prendas balanceadas y sentimientos balanceados, se espera que cada clase conjunta reúna cerca del 5 % de los pares. Se grafica el número de pares por clase en entrenamiento.

In [ ]:
# ============================================================
# Pares por clase conjunta en entrenamiento
# ============================================================
conteo = np.bincount(Y_CONJUNTA["train"], minlength=20)
fig = go.Figure(go.Bar(x=list(range(20)), y=conteo, marker_color=[COLORES[0] if i % 2 else COLORES[2] for i in range(20)]))
fig.update_xaxes(tickvals=list(range(20)), ticktext=CLASES_CONJUNTAS, tickangle=-60)
fig.update_layout(title="Pares por clase conjunta (entrenamiento) · azul oscuro: positiva, azul claro: negativa",
                  yaxis_title="pares", height=480)
print(f"Mínimo: {conteo.min()} ({conteo.min() / conteo.sum() * 100:.2f} %) | máximo: {conteo.max()} ({conteo.max() / conteo.sum() * 100:.2f} %)")
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Proporción mínima y máxima de las clases conjuntas: ¿se necesita corregir el desbalance?
- Usos por reseña en cada partición (sección 6.1).

### 6.3 Datos de entrada de la fusión

Para cada par se toma la representación de su imagen y la de su reseña. Los vectores se **estandarizan** opcionalmente (restando la media y dividiendo por la desviación estándar de cada componente, calculadas solo con los pares de entrenamiento), porque la CNN y el modelo de texto producen activaciones en escalas distintas y una modalidad podría dominar a la otra al concatenarlas. La estandarización es opcional para poder medir su efecto con un experimento.

`DATOS` reúne, por partición, los vectores de imagen (`img`), de texto (`txt`) y la clase conjunta (`y`), en GPU.

In [ ]:
# ============================================================
# Vectores de entrada de la fusión y estadísticos de estandarización
# ============================================================
DATOS = {p: {"img": REP_IMAGEN[p], "txt": REP_TEXTO[p][torch.as_tensor(PARES[p], device=device)],
             "y": torch.as_tensor(Y_CONJUNTA[p], device=device)} for p in PARES}
ESTADISTICOS = {m: (DATOS["train"][m].mean(0), DATOS["train"][m].std(0) + 1e-6) for m in ["img", "txt"]}
display(pd.DataFrame({"imagen": [DATOS["train"]["img"].mean().item(), DATOS["train"]["img"].std().item()],
                      "texto": [DATOS["train"]["txt"].mean().item(), DATOS["train"]["txt"].std().item()]},
                     index=["media", "desviación"]).round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Diferencia de escala entre las representaciones de imagen y de texto.

## 7. Líneas base

- **Clase mayoritaria:** predice siempre la clase conjunta más frecuente del entrenamiento (≈ 5 % de exactitud).
- **Fusión tardía:** combina las predicciones de las cabezas originales de ambos modelos, sin entrenar nada: prenda predicha por la CNN × 2 + sentimiento predicho por el modelo de texto. Como prenda y sentimiento son independientes en estos pares, si los errores de ambas ramas también lo son, su exactitud conjunta se acerca al **producto** de las exactitudes de cada rama.

Un clasificador fusionado que no supere a la clase mayoritaria no aprendió; uno que se acerque a la fusión tardía aprovecha casi toda la información disponible. Las líneas base que usan **una sola modalidad** necesitan entrenar el clasificador, así que se obtienen en el primer experimento.

`metricas(y_true, y_pred)` retorna un diccionario con exactitud, precisión macro, exhaustividad macro, F1 macro y F1 ponderado.

In [ ]:
# ============================================================
# Métricas y líneas base sin entrenamiento
# ============================================================
def metricas(y_true, y_pred):
    precision, exhaustividad, _, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"exactitud": accuracy_score(y_true, y_pred), "precision_macro": precision,
            "exhaustividad_macro": exhaustividad, "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "f1_ponderado": f1_score(y_true, y_pred, average="weighted")}


def fusion_tardia(p):
    return PRED_PRENDA[p] * 2 + PRED_SENTIMIENTO[p][PARES[p]]


CLASE_MAYORITARIA = int(np.bincount(Y_CONJUNTA["train"]).argmax())
producto_val = accuracy_score(Y_PRENDA["val"], PRED_PRENDA["val"]) * accuracy_score(Y_SENTIMIENTO["val"][PARES["val"]], PRED_SENTIMIENTO["val"][PARES["val"]])
tabla_lineas_base_val = pd.DataFrame([
    {"modelo": "Clase mayoritaria", **metricas(Y_CONJUNTA["val"], np.full(len(Y_CONJUNTA["val"]), CLASE_MAYORITARIA))},
    {"modelo": "Fusión tardía (cabezas originales)", **metricas(Y_CONJUNTA["val"], fusion_tardia("val"))},
])
print(f"Producto de las exactitudes de cada rama en los pares de validación: {producto_val:.4f}")
display(tabla_lineas_base_val.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Exactitud de la fusión tardía frente al producto de las exactitudes de cada rama: ¿los errores de ambas ramas son independientes?
- F1 macro de la fusión tardía: la referencia que debe alcanzar el clasificador fusionado.

## 8. Utilidades de entrenamiento, evaluación y experimentos

### 8.1 Preparación de los lotes, una época y predicción

- `preparar(datos, indices, config)`: toma los vectores de imagen y de texto de los pares indicados y los estandariza si `config["normalizar"]` es verdadero. Retorna ambos tensores.
- `construir_criterio(nombre)`: entropía cruzada, o entropía cruzada con *label smoothing* de 0,1.
- `entrenar_epoca(modelo, datos, config, optimizador, criterio, generador)`: recorre los pares en orden aleatorio, calcula la pérdida, hace *backpropagation* y actualiza los pesos. Retorna la pérdida y la exactitud de entrenamiento.
- `predecir(modelo, datos, config, criterio)`: evalúa sin gradientes en lotes de 4.096 pares. Retorna la pérdida (si se entrega un criterio), las etiquetas reales, las predichas y las probabilidades.

In [ ]:
# ============================================================
# Preparación de lotes, pérdida, época y predicción
# ============================================================
def preparar(datos, indices, config):
    img, txt = datos["img"][indices], datos["txt"][indices]
    if config["normalizar"]:
        img = (img - ESTADISTICOS["img"][0]) / ESTADISTICOS["img"][1]
        txt = (txt - ESTADISTICOS["txt"][0]) / ESTADISTICOS["txt"][1]
    return img, txt


def construir_criterio(nombre):
    return nn.CrossEntropyLoss(label_smoothing=0.1 if nombre == "suavizado_etiquetas" else 0.0)


def entrenar_epoca(modelo, datos, config, optimizador, criterio, generador):
    modelo.train()
    perdida, aciertos, total = 0.0, 0, 0
    orden = torch.randperm(len(datos["y"]), generator=generador, device=device)
    for inicio in range(0, len(orden), config["batch_size"]):
        indices = orden[inicio:inicio + config["batch_size"]]
        img, txt = preparar(datos, indices, config)
        y = datos["y"][indices]
        optimizador.zero_grad()
        logits = modelo(img, txt)
        loss = criterio(logits, y)
        loss.backward()
        optimizador.step()
        perdida += loss.item() * len(y)
        aciertos += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return perdida / total, aciertos / total


def predecir(modelo, datos, config, criterio=None, lote=4096):
    modelo.eval()
    salidas, perdida = [], 0.0
    with torch.no_grad():
        for inicio in range(0, len(datos["y"]), lote):
            indices = torch.arange(inicio, min(inicio + lote, len(datos["y"])), device=device)
            logits = modelo(*preparar(datos, indices, config))
            if criterio is not None:
                perdida += criterio(logits, datos["y"][indices]).item() * len(indices)
            salidas.append(logits)
    logits = torch.cat(salidas)
    return {"perdida": perdida / len(datos["y"]), "y_true": datos["y"].cpu().numpy(),
            "y_pred": logits.argmax(1).cpu().numpy(), "probas": torch.softmax(logits, 1).cpu().numpy()}

### 8.2 Entrenamiento con Early Stopping y registro de métricas

`entrenar_y_registrar(config, semilla, epocas, paciencia, verbose, forzar)` construye el clasificador fusionado y lo entrena con **Early Stopping** sobre la pérdida de validación, restaurando los pesos de la mejor época. Con `paciencia=0` entrena todas las épocas. Retorna `(fila, historia, modelo)`.

- **Régimen uniforme:** `EPOCAS_EXPERIMENTO = 30` y `PACIENCIA_EXPERIMENTO = 5`, incluido el modelo final. Cada época procesa solo vectores de 128 valores, así que el tope puede ser más holgado que en los modelos de imagen y de texto.
- **Extensión sin reiniciar:** si la mejor época queda en las dos últimas del tope, la corrida continúa hasta `EPOCAS_EXTENDIDAS = 50`.
- La pérdida de validación se mide siempre con la entropía cruzada sin suavizado (`CRITERIO_EVAL`), para que sea comparable entre variantes.
- Si existe un resultado con la misma clave y `forzar=False`, guardado en el JSON o entrenado antes en la misma sesión, se reutiliza y no se entrena.
- Cada corrida nueva se guarda en el JSON y se imprime como `CORRIDA_GUARDADA {...}`.
- Si la pérdida deja de ser finita, la corrida se detiene, se marca con `diverge = True` y se registra como **Error / Falla**.

In [ ]:
# ============================================================
# Entrenamiento con Early Stopping, extensión y registro de métricas
# ============================================================
EPOCAS_EXPERIMENTO = 30
PACIENCIA_EXPERIMENTO = 5
EPOCAS_EXTENDIDAS = 50
CRITERIO_EVAL = nn.CrossEntropyLoss()


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


def clave_corrida(config, semilla, epocas, paciencia):
    return json.dumps({**config, "semilla": semilla, "epocas": epocas, "paciencia": paciencia}, sort_keys=True)


def entrenar_y_registrar(config, semilla=SEED, epocas=None, paciencia=None, verbose=False, forzar=False):
    epocas = epocas or EPOCAS_EXPERIMENTO
    paciencia = PACIENCIA_EXPERIMENTO if paciencia is None else paciencia
    clave = clave_corrida(config, semilla, epocas, paciencia)
    guardadas = {**RESULTADOS_GUARDADOS, **CORRIDAS_NUEVAS}
    if not forzar and clave in guardadas:
        return {**guardadas[clave], "origen": "guardado"}, None, None

    fijar_semilla(semilla)
    generador = torch.Generator(device=device).manual_seed(semilla)
    modelo = construir_fusion(config)
    if forzar and REUTILIZAR_MODELOS:
        guardada = cargar_modelo_corrida(clave, modelo)
        if guardada is not None:
            fila, historia = guardada
            print(f"Corrida cargada desde {nombre_modelo_corrida(clave)} (sin entrenar)")
            return {**fila, "origen": "modelo guardado"}, historia, modelo
    criterio = construir_criterio(config["perdida"])
    optimizador = torch.optim.Adam(modelo.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    historia = {"loss_train": [], "exactitud_train": [], "loss_val": [], "exactitud_val": [], "f1_val": []}
    mejor_loss, mejor_estado, mejor_epoca, sin_mejora, diverge = float("inf"), None, 0, 0, False
    tope, epoca, inicio = epocas, 0, time.time()
    while epoca < tope:
        loss_tr, acc_tr = entrenar_epoca(modelo, DATOS["train"], config, optimizador, criterio, generador)
        val = predecir(modelo, DATOS["val"], config, CRITERIO_EVAL)
        acc_val = accuracy_score(val["y_true"], val["y_pred"])
        f1_val = f1_score(val["y_true"], val["y_pred"], average="macro")
        for k, valor in zip(historia, [loss_tr, acc_tr, val["perdida"], acc_val, f1_val]):
            historia[k].append(valor)
        if verbose:
            print(f"Época {epoca + 1:02} | loss train {loss_tr:.4f} | loss val {val['perdida']:.4f} "
                  f"| exactitud val {acc_val:.4f} | F1 macro val {f1_val:.4f}")
        if not (np.isfinite(loss_tr) and np.isfinite(val["perdida"])):
            print(f"  Error / Falla: la pérdida dejó de ser finita en la época {epoca + 1} (el entrenamiento divergió).")
            diverge = True
            break
        if val["perdida"] < mejor_loss:
            mejor_loss, mejor_epoca, sin_mejora = val["perdida"], epoca, 0
            mejor_estado = copy.deepcopy(modelo.state_dict())
        else:
            sin_mejora += 1
            if paciencia and sin_mejora >= paciencia:
                break
        epoca += 1
        if epoca == tope and paciencia and mejor_epoca >= tope - 2 and tope < EPOCAS_EXTENDIDAS:
            print(f"  La mejor época ({mejor_epoca + 1}) está en el tope: se continúa hasta {EPOCAS_EXTENDIDAS} épocas.")
            tope = EPOCAS_EXTENDIDAS
    duracion = time.time() - inicio
    if mejor_estado is None:
        mejor_estado = copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(mejor_estado)
    if paciencia and not diverge and mejor_epoca >= tope - 2:
        print(f"  Aviso: la mejor época ({mejor_epoca + 1}) sigue al final del tope extendido: la variante no convergió.")

    m = mejor_epoca
    fila = {
        "exactitud_train": historia["exactitud_train"][m],
        "exactitud_val": historia["exactitud_val"][m],
        "f1_macro_val": historia["f1_val"][m],
        "loss_val": historia["loss_val"][m],
        "brecha_train_val": historia["exactitud_train"][m] - historia["exactitud_val"][m],
        "mejor_epoca": m + 1,
        "epocas_entrenadas": len(historia["loss_train"]),
        "parametros": contar_parametros(modelo),
        "segundos": duracion,
        "segundos_por_epoca": duracion / len(historia["loss_train"]),
        "diverge": diverge,
    }
    CORRIDAS_NUEVAS[clave] = fila
    if forzar:
        guardar_modelo_corrida(clave, modelo, historia, fila)
    print("CORRIDA_GUARDADA " + json.dumps({"clave": clave, "fila": fila}, default=float))
    guardar_corridas()
    return {**fila, "origen": "entrenado"}, historia, modelo

### 8.3 Experimentos controlados, selección y ampliación del rango

- `ejecutar_experimento(nombre, variantes, config_actual, forzar)` entrena (o recupera) las **variantes** `{etiqueta: cambios}`, que modifican un solo hiperparámetro respecto de la configuración actual. Retorna una tabla ordenada por **F1 macro de validación**, con la diferencia respecto de la mejor y si cae dentro del rango de error. Las historias y los modelos de las variantes entrenadas quedan en `historias_ultimo_experimento` y `modelos_ultimo_experimento`. Si se repite un experimento, la configuración vuelve a su estado previo y se descartan las decisiones posteriores.
- `ampliar_rango(nombre, variantes, config_actual, hiperparametro, arriba, abajo, minimo, maximo)`: si la mejor variante queda en el **borde** del rango probado, agrega el valor siguiente en esa dirección y lo entrena, mientras el ganador siga en el borde y no se supere el límite. Retorna la tabla actualizada.
- `seleccionar_variante(tabla)`: **propone** la mejor variante y lista las que están en empate técnico. Los empates se revisan caso a caso y la decisión queda en una línea `# Cambio Manual`.
- `aplicar_variante(nombre, variantes, elegida)`: incorpora la variante a `CONFIG_ACTUAL`, registra la decisión con su F1 y el de la configuración previa, y muestra la configuración.
- `tabla_de(nombre)`: retorna la tabla guardada de un experimento.

In [ ]:
# ============================================================
# Experimentos controlados (un hiperparámetro a la vez)
# ============================================================
resultados_experimentos = []
historial_experimentos = []
decisiones = {}
historias_ultimo_experimento = {}
modelos_ultimo_experimento = {}
COLUMNAS_TABLA = ["variante", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca",
                  "epocas_entrenadas", "parametros", "segundos_por_epoca", "diferencia_vs_mejor", "dentro_del_margen",
                  "diverge", "origen"]


def entrenar_variantes(nombre, variantes, config_actual, forzar=False):
    filas = []
    for etiqueta, cambios in variantes.items():
        fila, historia, modelo = entrenar_y_registrar({**config_actual, **cambios}, forzar=forzar)
        print(f"[{nombre}] {etiqueta}: {fila['origen']}")
        if historia is not None:
            historias_ultimo_experimento[etiqueta] = historia
            modelos_ultimo_experimento[etiqueta] = modelo
        filas.append({"experimento": nombre, "variante": etiqueta, **fila})
    return filas


def armar_tabla(filas):
    tabla = pd.DataFrame(filas)
    tabla["diferencia_vs_mejor"] = tabla["f1_macro_val"].max() - tabla["f1_macro_val"]
    tabla["dentro_del_margen"] = tabla["diferencia_vs_mejor"] <= MARGEN_ERROR
    return tabla.sort_values("f1_macro_val", ascending=False).reset_index(drop=True)


def ejecutar_experimento(nombre, variantes, config_actual, forzar=False):
    nombres = [n for n, _ in historial_experimentos]
    if nombre in nombres:
        i = nombres.index(nombre)
        config_actual.clear()
        config_actual.update(historial_experimentos[i][1])
        for n in nombres[i:]:
            decisiones.pop(n, None)
        del historial_experimentos[i:]
        del resultados_experimentos[i:]
        print(f"[{nombre}] Experimento repetido: la configuración vuelve a su estado previo "
              f"y se descartan las decisiones posteriores ({', '.join(nombres[i:])}).")
    historial_experimentos.append((nombre, dict(config_actual)))
    historias_ultimo_experimento.clear()
    modelos_ultimo_experimento.clear()
    tabla = armar_tabla(entrenar_variantes(nombre, variantes, config_actual, forzar))
    resultados_experimentos.append(tabla)
    return tabla


def ampliar_rango(nombre, variantes, config_actual, hiperparametro, arriba, abajo, minimo, maximo):
    tabla = resultados_experimentos[-1]
    while True:
        valores = {etq: cambios[hiperparametro] for etq, cambios in variantes.items()}
        mejor = valores[tabla.loc[0, "variante"]]
        if mejor == max(valores.values()) and arriba(mejor) <= maximo:
            nuevo = arriba(mejor)
        elif mejor == min(valores.values()) and abajo(mejor) >= minimo:
            nuevo = abajo(mejor)
        else:
            return tabla
        etiqueta = f"{nuevo:g}" if isinstance(nuevo, float) else str(nuevo)
        print(f"[{nombre}] La mejor variante está en el borde del rango: se agrega {hiperparametro} = {etiqueta}")
        variantes[etiqueta] = {hiperparametro: nuevo}
        filas = tabla.drop(columns=["diferencia_vs_mejor", "dentro_del_margen"]).to_dict("records")
        filas += entrenar_variantes(nombre, {etiqueta: {hiperparametro: nuevo}}, config_actual)
        tabla = armar_tabla(filas)
        resultados_experimentos[-1] = tabla


def seleccionar_variante(tabla):
    mejor = tabla.loc[0, "variante"]
    empatadas = tabla.loc[tabla["dentro_del_margen"] & (tabla["variante"] != mejor), "variante"].tolist()
    print(f"Mejor F1 macro de validación: {mejor}")
    if empatadas:
        print(f"En empate técnico (dentro de {MARGEN_ERROR * 100:.2f} pp): {', '.join(map(str, empatadas))}")
    return mejor


def tabla_de(nombre):
    return resultados_experimentos[[n for n, _ in historial_experimentos].index(nombre)]


def aplicar_variante(nombre, variantes, elegida):
    tabla, config_previa = resultados_experimentos[-1], historial_experimentos[-1][1]
    previa = [etq for etq, cambios in variantes.items() if all(config_previa.get(k) == v for k, v in cambios.items())]
    f1_de = dict(zip(tabla["variante"], tabla["f1_macro_val"]))
    decisiones[nombre] = {"elegida": elegida, "f1_elegida": f1_de[elegida],
                          "variante_previa": previa[0] if previa else None,
                          "f1_previa": f1_de[previa[0]] if previa else None}
    CONFIG_ACTUAL.update(variantes[elegida])
    mostrar_config(CONFIG_ACTUAL)

### 8.4 Curvas de entrenamiento y configuración

- `graficar_curvas(historia, titulo)`: pérdida, exactitud y F1 macro de validación por época en tres paneles horizontales, con la mejor época marcada.
- `graficar_comparacion(historias, clave, titulo, eje_y)`: una línea por variante para la métrica `clave`.
- `mostrar_config(config)`: muestra la configuración como tabla vertical.

In [ ]:
# ============================================================
# Visualización de curvas y configuración
# ============================================================
def graficar_curvas(historia, titulo):
    epocas = list(range(1, len(historia["loss_train"]) + 1))
    mejor = int(np.argmin(historia["loss_val"])) + 1
    fig = make_subplots(rows=1, cols=3, subplot_titles=["Pérdida", "Exactitud", "F1 macro (validación)"])
    for col, tr, va in [(1, "loss_train", "loss_val"), (2, "exactitud_train", "exactitud_val")]:
        fig.add_trace(go.Scatter(x=epocas, y=historia[tr], name="entrenamiento", line_color=COLORES[2],
                                 legendgroup="train", showlegend=col == 1), row=1, col=col)
        fig.add_trace(go.Scatter(x=epocas, y=historia[va], name="validación", line_color=COLORES[0],
                                 legendgroup="val", showlegend=col == 1), row=1, col=col)
    fig.add_trace(go.Scatter(x=epocas, y=historia["f1_val"], name="F1 macro validación", line_color=COLORES[1]), row=1, col=3)
    for col in [1, 2, 3]:
        fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
        fig.update_xaxes(title_text="época", row=1, col=col)
    fig.update_layout(title=f"{titulo} · mejor época: {mejor}", height=420)
    fig.show()


def graficar_comparacion(historias, clave, titulo, eje_y):
    paleta = COLORES + ["#0E2747", "#8A6A00", "#52514E"]
    fig = go.Figure()
    for i, (etiqueta, historia) in enumerate(historias.items()):
        fig.add_trace(go.Scatter(x=list(range(1, len(historia[clave]) + 1)), y=historia[clave], name=etiqueta,
                                 line_color=paleta[i % len(paleta)]))
    fig.update_layout(title=titulo, xaxis_title="época", yaxis_title=eje_y, height=420)
    fig.show()


def mostrar_config(config):
    display(pd.DataFrame({"valor": pd.Series(config).astype(str)}))

## 9. Arquitecturas de fusión

### 9.1 Concatenación + MLP y atención entre modalidades

**`FusionConcatenacion`.** Une los vectores de las modalidades indicadas en `modalidades` (`ambas`: 256 valores; `imagen` o `texto`: 128) y los clasifica con `capas` capas densas de `unidades` neuronas, cada una con ReLU y Dropout, seguidas de la salida de 20 *logits*. Con una sola modalidad sirve como línea base: mide cuánto se puede predecir sin la otra.

**`FusionAtencion`.** Proyecta cada representación a `unidades` valores y la trata como un *token*; a cada uno se le suma un *embedding* de modalidad aprendido, para que el modelo distinga cuál es la imagen y cuál el texto. Los dos *tokens* pasan por un `EncoderBlock` (el mismo bloque de autoatención con *feed-forward*, conexión residual y *LayerNorm* del Transformer de texto, con `cabezas` cabezas). En la autoatención, el *token* de la imagen toma información del *token* del texto y viceversa, con pesos que dependen de cada par. Luego se concatenan ambos *tokens* y se clasifican con el mismo MLP de la concatenación. Los pesos de atención quedan disponibles para analizar cuánto mira cada modalidad a la otra.

Ambas clases retornan los 20 *logits* desde `forward(img, txt)`. `construir_fusion(config)` crea la arquitectura indicada en `config["tecnica"]` en el dispositivo.

In [ ]:
# ============================================================
# Clasificadores fusionados
# ============================================================
def mlp(entrada, config):
    capas = []
    for _ in range(config["capas"]):
        capas += [nn.Linear(entrada, config["unidades"]), nn.ReLU(), nn.Dropout(config["dropout"])]
        entrada = config["unidades"]
    return nn.Sequential(*capas, nn.Linear(entrada, 20))


class FusionConcatenacion(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.modalidades = config["modalidades"]
        entrada = UNIDADES_REPRESENTACION * (2 if self.modalidades == "ambas" else 1)
        self.clasificador = mlp(entrada, config)

    def forward(self, img, txt):
        x = {"ambas": lambda: torch.cat([img, txt], dim=1), "imagen": lambda: img, "texto": lambda: txt}[self.modalidades]()
        return self.clasificador(x)


class FusionAtencion(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config["unidades"]
        self.proyeccion_img = nn.Linear(UNIDADES_REPRESENTACION, d)
        self.proyeccion_txt = nn.Linear(UNIDADES_REPRESENTACION, d)
        self.modalidad = nn.Parameter(torch.randn(1, 2, d) * 0.02)
        self.bloque = EncoderBlock(d, config["cabezas"], 2 * d, config["dropout"])
        self.clasificador = mlp(2 * d, config)

    def forward(self, img, txt):
        tokens = torch.stack([self.proyeccion_img(img), self.proyeccion_txt(txt)], dim=1) + self.modalidad
        # tokens = [batch size, 2, unidades]
        tokens = self.bloque(tokens, None)
        return self.clasificador(tokens.flatten(1))


def construir_fusion(config):
    clase = FusionAtencion if config["tecnica"] == "atencion" else FusionConcatenacion
    return clase(config).to(device)

### 9.2 Configuración base

| Hiperparámetro | Valor base | Justificación |
|---|---|---|
| `tecnica` | `concatenacion` | La fusión más simple: no agrega interacción entre modalidades |
| `modalidades` | `ambas` | La clase conjunta necesita la prenda y el sentimiento |
| `normalizar` | `True` | Las representaciones de imagen y de texto vienen de redes distintas y pueden tener escalas distintas (sección 6.3) |
| `capas` | 2 | Suficiente para combinar ambas representaciones de forma no lineal |
| `unidades` | 256 | Igual a la entrada concatenada (128 + 128) |
| `cabezas` | 4 | Solo para la atención: cabezas de 64 dimensiones con 256 unidades |
| `dropout` | 0,25 | Regularización moderada, igual que en la CNN |
| `perdida` | `entropia_cruzada` | Función estándar para clasificación multiclase |
| `learning_rate` | 0,001 | Valor por defecto de Adam (Kingma y Ba, 2015) |
| `batch_size` | 256 | 196 actualizaciones por época con 50.000 pares; los vectores son pequeños y caben lotes grandes |
| `weight_decay` | 0 | Base sin penalización L2 |

In [ ]:
# ============================================================
# Configuración base
# ============================================================
CONFIG_BASE = dict(tecnica="concatenacion", modalidades="ambas", normalizar=True, capas=2, unidades=256, cabezas=4,
                   dropout=0.25, perdida="entropia_cruzada", learning_rate=0.001, batch_size=256, weight_decay=0.0)
CONFIG_ACTUAL = dict(CONFIG_BASE)
mostrar_config(CONFIG_BASE)

## 10. Modelo base y rango de error

### 10.1 Entrenamiento del modelo base

Se entrena la configuración base con la semilla 42, mostrando el avance por época, o se carga desde su `.pt` si ya se entrenó (sección 2.1). Es la referencia de todos los experimentos y la primera de las 5 semillas del rango de error.

In [ ]:
# ============================================================
# Entrenamiento del modelo base
# ============================================================
fila_base, historia_base, modelo_base = entrenar_y_registrar(CONFIG_BASE, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Estructura, parámetros y métricas del modelo base
# ============================================================
print(modelo_base)
print(f"Parámetros entrenables: {contar_parametros(modelo_base):,}")
display(pd.DataFrame([fila_base]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo base
# ============================================================
graficar_curvas(historia_base, "Modelo base (concatenación + MLP)")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de validación frente a la clase mayoritaria y a la fusión tardía (sección 7).
- Mejor época y brecha entrenamiento–validación.

### 10.2 Rango de error entre reentrenamientos

Se reentrena `CONFIG_BASE` con otras cuatro semillas (5 en total) y se calcula el **rango** (máximo − mínimo) del F1 macro de validación. Ese valor queda en `MARGEN_ERROR` y se usa en todo el notebook: dos variantes cuya diferencia sea menor o igual se consideran en **empate técnico**. Las semillas cambian los pesos iniciales y el orden de los lotes; los pares se mantienen fijos.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración base con distintas semillas
# ============================================================
SEMILLAS_ADICIONALES = [1, 2, 3, 4]

filas_variabilidad = [{"semilla": SEED, **fila_base}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_BASE, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_variabilidad.append({"semilla": semilla, **fila})
tabla_variabilidad = pd.DataFrame(filas_variabilidad)

In [ ]:
# ============================================================
# Resultados por semilla
# ============================================================
display(tabla_variabilidad.round(4))

In [ ]:
# ============================================================
# Estadísticos de variación y rango de error
# ============================================================
columnas_variacion = ["f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca"]
estadisticos_variabilidad = tabla_variabilidad[columnas_variacion].agg(["mean", "std", "min", "max"]).T
estadisticos_variabilidad["rango"] = estadisticos_variabilidad["max"] - estadisticos_variabilidad["min"]
MARGEN_ERROR = estadisticos_variabilidad.loc["f1_macro_val", "rango"]
print(f"Rango de error entre reentrenamientos (MARGEN_ERROR): {MARGEN_ERROR:.4f} ({MARGEN_ERROR * 100:.2f} pp)")
display(estadisticos_variabilidad.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Media, desviación estándar y rango del F1 macro con 5 semillas: el valor de MARGEN_ERROR en puntos porcentuales.

## 11. Experimento clave: aporte de cada modalidad

Se entrena el mismo clasificador con solo la representación de la imagen, solo la del texto y ambas. Con una sola modalidad, el modelo puede acertar la mitad de la clase conjunta (la prenda o el sentimiento) y debe adivinar la otra mitad, así que se esperan valores cercanos a «exactitud de la rama × 1/2» con la imagen y a «exactitud de la rama × 1/10» con el texto. La configuración mantiene `ambas`, porque la tarea las requiere; el experimento mide cuánto aporta la fusión. Como se evalúan también en test, junto con el modelo final, cada uno se entrena o, si ya existe su `.pt`, se carga desde él (sección 2.1); no se reutiliza solo con las métricas.

### 11.1 Solo imagen, solo texto y ambas

Variantes: concatenación + MLP con cada modalidad por separado y con ambas.

In [ ]:
# ============================================================
# Experimento 11.1: solo imagen, solo texto y ambas
# ============================================================
variantes_modalidades = {"solo imagen": dict(modalidades="imagen"), "solo texto": dict(modalidades="texto"), "ambas": dict(modalidades="ambas")}
tabla_modalidades = ejecutar_experimento("modalidades", variantes_modalidades, CONFIG_ACTUAL, forzar=True)
modelos_modalidades = dict(modelos_ultimo_experimento)

In [ ]:
# ============================================================
# Resultados del experimento 11.1
# ============================================================
display(tabla_modalidades[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 11.1
# ============================================================
elegida = "ambas"  # requerida por el enunciado: el experimento aporta evidencia, no decide
aplicar_variante("modalidades", variantes_modalidades, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de cada modalidad sola frente a la fusión: aporte de la fusión en puntos porcentuales.
- ¿Las cifras con una sola modalidad se acercan a lo esperado (mitad o décima parte de la exactitud de la rama)?

## 12. Experimento clave: técnica de fusión

Se compara la concatenación + MLP con la atención entre modalidades (sección 9.1), con la configuración vigente. La atención permite que la combinación dependa de cada par; la concatenación aplica siempre la misma combinación lineal en la primera capa. Como prenda y sentimiento son independientes en estos pares, no se espera que la atención encuentre interacciones que la concatenación no capte: si ambas empatan, la más simple es la candidata natural, y se decide por criterio contextual. Para graficar sus curvas y analizar los pesos de atención, cada variante se entrena o, si ya existe su `.pt`, se carga desde él (sección 2.1); no se reutiliza solo con las métricas.

### 12.1 Concatenación frente a atención

Variantes: concatenación + MLP y atención entre modalidades.

In [ ]:
# ============================================================
# Experimento 12.1: concatenación frente a atención
# ============================================================
variantes_tecnica = {"concatenación + MLP": dict(tecnica="concatenacion"), "atención entre modalidades": dict(tecnica="atencion")}
tabla_tecnica = ejecutar_experimento("técnica de fusión", variantes_tecnica, CONFIG_ACTUAL, forzar=True)
historias_tecnica = dict(historias_ultimo_experimento)
modelos_tecnica = dict(modelos_ultimo_experimento)

In [ ]:
# ============================================================
# Resultados del experimento 12.1
# ============================================================
display(tabla_tecnica[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Curvas de validación del experimento 12.1
# ============================================================
graficar_comparacion(historias_tecnica, "f1_val", "F1 macro de validación por época según la técnica de fusión", "F1 macro de validación")

In [ ]:
# ============================================================
# Selección de la variante del experimento 12.1
# ============================================================
elegida = seleccionar_variante(tabla_tecnica)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("técnica de fusión", variantes_tecnica, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Diferencia de F1 macro entre ambas técnicas, comparada con MARGEN_ERROR.
- Parámetros y segundos por época que agrega la atención.
- Si hay empate técnico, opciones y criterio (se decide por criterio contextual).

## 13. Experimentos de preprocesamiento y pérdida

Se mide el efecto de estandarizar las representaciones y de suavizar las etiquetas.

### 13.1 Estandarización de las representaciones

Sin estandarizar, la modalidad con activaciones más grandes domina las primeras actualizaciones. Se compara con y sin estandarización.

In [ ]:
# ============================================================
# Experimento 13.1: estandarización de las representaciones
# ============================================================
variantes_normalizar = {"estandarizadas": dict(normalizar=True), "sin estandarizar": dict(normalizar=False)}
tabla_normalizar = ejecutar_experimento("normalización", variantes_normalizar, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 13.1
# ============================================================
display(tabla_normalizar[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.1
# ============================================================
elegida = seleccionar_variante(tabla_normalizar)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("normalización", variantes_normalizar, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

### 13.2 Función de pérdida

Con 20 clases y vectores fijos, el clasificador puede volverse sobreconfiado. Se compara la entropía cruzada con la misma pérdida con suavizado de etiquetas de 0,1 (Szegedy et al., 2016). La pérdida de validación de la tabla se mide siempre sin suavizado.

In [ ]:
# ============================================================
# Experimento 13.2: función de pérdida
# ============================================================
variantes_perdida = {"entropía cruzada": dict(perdida="entropia_cruzada"), "suavizado de etiquetas": dict(perdida="suavizado_etiquetas")}
tabla_perdida = ejecutar_experimento("pérdida", variantes_perdida, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 13.2
# ============================================================
display(tabla_perdida[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.2
# ============================================================
elegida = seleccionar_variante(tabla_perdida)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("pérdida", variantes_perdida, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

## 14. Experimentos de entrenamiento y arquitectura

Tasa de aprendizaje, tamaño del lote, épocas y capacidad del clasificador fusionado.

### 14.1 Tasa de aprendizaje

Se prueban 0,0003, 0,001 y 0,003; si la mejor queda en un borde, se amplía hasta 0,0001 o 0,01.

In [ ]:
# ============================================================
# Experimento 14.1: tasa de aprendizaje
# ============================================================
variantes_tasa = {"0.0003": dict(learning_rate=0.0003), "0.001": dict(learning_rate=0.001), "0.003": dict(learning_rate=0.003)}
tabla_tasa = ejecutar_experimento("tasa de aprendizaje", variantes_tasa, CONFIG_ACTUAL)
tabla_tasa = ampliar_rango("tasa de aprendizaje", variantes_tasa, CONFIG_ACTUAL, "learning_rate", lambda v: round(v * 3, 6), lambda v: round(v / 3, 6), 0.0001, 0.01)

In [ ]:
# ============================================================
# Resultados del experimento 14.1
# ============================================================
display(tabla_tasa[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 14.1
# ============================================================
elegida = seleccionar_variante(tabla_tasa)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("tasa de aprendizaje", variantes_tasa, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

### 14.2 Tamaño del lote

Se prueban 128, 256 y 512 con la tasa vigente; si la mejor queda en un borde, se amplía entre 32 y 2.048.

In [ ]:
# ============================================================
# Experimento 14.2: tamaño del lote
# ============================================================
variantes_lote = {"128": dict(batch_size=128), "256": dict(batch_size=256), "512": dict(batch_size=512)}
tabla_lote = ejecutar_experimento("tamaño del lote", variantes_lote, CONFIG_ACTUAL)
tabla_lote = ampliar_rango("tamaño del lote", variantes_lote, CONFIG_ACTUAL, "batch_size", lambda v: v * 2, lambda v: v // 2, 32, 2048)

In [ ]:
# ============================================================
# Resultados del experimento 14.2
# ============================================================
display(tabla_lote[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 14.2
# ============================================================
elegida = seleccionar_variante(tabla_lote)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("tamaño del lote", variantes_lote, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

### 14.3 Épocas sin Early Stopping

Para justificar el tope de 30 épocas y la paciencia de 5, se entrena la configuración vigente durante 60 épocas **sin Early Stopping** y se observa en qué época la pérdida de validación empieza a subir. Esta corrida no cambia la configuración.

In [ ]:
# ============================================================
# Experimento 14.3: 60 épocas sin Early Stopping
# ============================================================
EPOCAS_SIN_ES = 60
fila_epocas, historia_epocas, _ = entrenar_y_registrar(CONFIG_ACTUAL, epocas=EPOCAS_SIN_ES, paciencia=0, forzar=True)

In [ ]:
# ============================================================
# Curvas sin Early Stopping
# ============================================================
graficar_curvas(historia_epocas, "Configuración vigente, 60 épocas sin Early Stopping")

In [ ]:
# ============================================================
# Métricas de validación en épocas seleccionadas
# ============================================================
epocas_revisadas = [5, 10, 20, 30, 45, 60]
tabla_epocas = pd.DataFrame({k: [historia_epocas[k][e - 1] for e in epocas_revisadas]
                             for k in ["loss_train", "loss_val", "exactitud_val", "f1_val"]}, index=epocas_revisadas)
tabla_epocas.index.name = "época"
print(f"Mejor época por pérdida de validación: {int(np.argmin(historia_epocas['loss_val'])) + 1}")
display(tabla_epocas.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Época de menor pérdida de validación y desde cuándo sube (sobreajuste).
- ¿El tope de 30 épocas con paciencia 5 es suficiente?

### 14.4 Número de capas del clasificador

Se prueban 1, 2 y 3 capas ocultas; si gana 3, se amplía a 4.

In [ ]:
# ============================================================
# Experimento 14.4: número de capas del clasificador
# ============================================================
variantes_capas = {"1": dict(capas=1), "2": dict(capas=2), "3": dict(capas=3)}
tabla_capas = ejecutar_experimento("capas", variantes_capas, CONFIG_ACTUAL)
tabla_capas = ampliar_rango("capas", variantes_capas, CONFIG_ACTUAL, "capas", lambda v: v + 1, lambda v: v - 1, 1, 4)

In [ ]:
# ============================================================
# Resultados del experimento 14.4
# ============================================================
display(tabla_capas[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 14.4
# ============================================================
elegida = seleccionar_variante(tabla_capas)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("capas", variantes_capas, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

### 14.5 Unidades por capa

Se prueban 128, 256 y 512 unidades; si la mejor queda en un borde, se amplía entre 64 y 1.024. Con la atención, `unidades` es también la dimensión de cada *token* y debe ser divisible por `cabezas`.

In [ ]:
# ============================================================
# Experimento 14.5: unidades por capa
# ============================================================
variantes_unidades = {"128": dict(unidades=128), "256": dict(unidades=256), "512": dict(unidades=512)}
tabla_unidades = ejecutar_experimento("unidades", variantes_unidades, CONFIG_ACTUAL)
tabla_unidades = ampliar_rango("unidades", variantes_unidades, CONFIG_ACTUAL, "unidades", lambda v: v * 2, lambda v: v // 2, 64, 1024)

In [ ]:
# ============================================================
# Resultados del experimento 14.5
# ============================================================
display(tabla_unidades[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 14.5
# ============================================================
elegida = seleccionar_variante(tabla_unidades)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("unidades", variantes_unidades, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

## 15. Experimentos de regularización

Se prueban por separado dos técnicas, cada una con al menos dos valores distintos de cero.

### 15.1 Dropout

Se prueban 0; 0,25 y 0,5; si gana 0,5, se amplía a 0,75.

In [ ]:
# ============================================================
# Experimento 15.1: dropout
# ============================================================
variantes_dropout = {"0.0": dict(dropout=0.0), "0.25": dict(dropout=0.25), "0.5": dict(dropout=0.5)}
tabla_dropout = ejecutar_experimento("dropout", variantes_dropout, CONFIG_ACTUAL)
tabla_dropout = ampliar_rango("dropout", variantes_dropout, CONFIG_ACTUAL, "dropout", lambda v: v + 0.25, lambda v: v - 0.25, 0.0, 0.75)

In [ ]:
# ============================================================
# Resultados del experimento 15.1
# ============================================================
display(tabla_dropout[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 15.1
# ============================================================
elegida = seleccionar_variante(tabla_dropout)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("dropout", variantes_dropout, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

### 15.2 Weight decay (penalización L2)

Se prueban 0; 0,00001 y 0,0001; si gana el mayor, se amplía a 0,001.

In [ ]:
# ============================================================
# Experimento 15.2: weight decay (penalización l2)
# ============================================================
variantes_weight_decay = {"0": dict(weight_decay=0.0), "1e-05": dict(weight_decay=1e-5), "0.0001": dict(weight_decay=1e-4)}
tabla_weight_decay = ejecutar_experimento("weight decay", variantes_weight_decay, CONFIG_ACTUAL)
tabla_weight_decay = ampliar_rango("weight decay", variantes_weight_decay, CONFIG_ACTUAL, "weight_decay", lambda v: v * 10, lambda v: v / 10 if v > 1e-5 else -1.0, 0.0, 0.001)

In [ ]:
# ============================================================
# Resultados del experimento 15.2
# ============================================================
display(tabla_weight_decay[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 15.2
# ============================================================
elegida = seleccionar_variante(tabla_weight_decay)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("weight decay", variantes_weight_decay, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide por criterio contextual).

## 16. Modelo final

### 16.1 Configuración final y tabla de decisiones

La configuración final es la que quedó en `CONFIG_ACTUAL`. La tabla de decisiones muestra, para cada experimento, la variante elegida, su F1 macro de validación y el de la variante que coincidía con la configuración previa.

In [ ]:
# ============================================================
# Configuración final y decisiones registradas
# ============================================================
CONFIG_FINAL = dict(CONFIG_ACTUAL)
tabla_decisiones = pd.DataFrame.from_dict(decisiones, orient="index")
tabla_decisiones.index.name = "experimento"
tabla_decisiones["aporte"] = tabla_decisiones["f1_elegida"] - tabla_decisiones["f1_previa"]
tabla_decisiones["supera_margen"] = tabla_decisiones["aporte"].abs() > MARGEN_ERROR
tabla_decisiones = tabla_decisiones.reset_index()
mostrar_config(CONFIG_FINAL)

In [ ]:
# ============================================================
# Tabla de decisiones
# ============================================================
display(tabla_decisiones.round(4))

### 16.2 Entrenamiento del modelo final

Se entrena la configuración final con la semilla 42 y el mismo régimen de épocas. Como su modelo se evalúa en test y se usa en la demostración, esta corrida se entrena o, si ya existe su `.pt`, se carga desde él (sección 2.1); no se reutiliza solo con las métricas.

In [ ]:
# ============================================================
# Entrenamiento del modelo final
# ============================================================
fila_final, historia_final, modelo_final = entrenar_y_registrar(CONFIG_FINAL, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Curvas del modelo final
# ============================================================
print(f"Parámetros entrenables: {contar_parametros(modelo_final):,}")
graficar_curvas(historia_final, "Modelo final de fusión")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de validación del modelo final frente al base, a la media de sus 5 semillas y a la fusión tardía.

## 17. Evaluación del modelo final en el conjunto de test

Los 10.000 pares de test (imágenes de test de Fashion-MNIST con reseñas de test de IMDB) no se usaron en ninguna decisión. Se evalúa una sola vez el modelo final. En esta tarea:

- **Exactitud conjunta:** proporción de pares con la prenda **y** el sentimiento correctos.
- **Precisión de una clase conjunta** (por ejemplo, «Camisa · positiva»): de los pares que el modelo asigna a esa clase, cuántos lo son. **Exhaustividad:** de los pares reales de esa clase, cuántos detecta.
- **F1 macro:** promedio del F1 de las 20 clases conjuntas.
- **Exactitud de cada componente:** se descompone la predicción conjunta en prenda (`clase // 2`) y sentimiento (`clase % 2`) para ver qué parte falla.

### 17.1 Métricas globales frente a las líneas base

Se comparan el modelo final, los modelos de una sola modalidad (sección 11), la fusión tardía y la clase mayoritaria, todos sobre los mismos pares de test.

In [ ]:
# ============================================================
# Predicción en test y métricas globales
# ============================================================
pred_test = predecir(modelo_final, DATOS["test"], CONFIG_FINAL, CRITERIO_EVAL)
pred_val = predecir(modelo_final, DATOS["val"], CONFIG_FINAL, CRITERIO_EVAL)
config_modalidad = {etq: {**historial_experimentos[0][1], **variantes_modalidades[etq]} for etq in modelos_modalidades}
filas_test = [{"modelo": "Clase mayoritaria", **metricas(Y_CONJUNTA["test"], np.full(len(Y_CONJUNTA["test"]), CLASE_MAYORITARIA))}]
filas_test += [{"modelo": f"Fusión base, {etq}", **metricas(Y_CONJUNTA["test"], predecir(m, DATOS["test"], config_modalidad[etq])["y_pred"])}
               for etq, m in modelos_modalidades.items() if etq != "ambas"]
filas_test += [{"modelo": "Fusión tardía (cabezas originales)", **metricas(Y_CONJUNTA["test"], fusion_tardia("test"))},
               {"modelo": "Fusión final", **metricas(pred_test["y_true"], pred_test["y_pred"]), "loss": pred_test["perdida"]}]
tabla_test = pd.DataFrame(filas_test)
producto_test = accuracy_score(Y_PRENDA["test"], PRED_PRENDA["test"]) * accuracy_score(Y_SENTIMIENTO["test"][PARES["test"]], PRED_SENTIMIENTO["test"][PARES["test"]])
print(f"Producto de las exactitudes de cada rama en test: {producto_test:.4f}")
display(tabla_test.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Exactitud y F1 macro de la fusión final frente a cada modalidad sola y a la fusión tardía (diferencias en puntos porcentuales).
- Diferencia entre validación y test: ¿el test confirma lo estimado en validación?

### 17.2 Descomposición en prenda y sentimiento

Se separa cada predicción conjunta en sus dos componentes y se compara su exactitud con la de cada rama por separado. Los errores se clasifican en: solo la prenda incorrecta, solo el sentimiento incorrecto o ambos.

In [ ]:
# ============================================================
# Exactitud por componente y tipos de error
# ============================================================
prenda_ok = pred_test["y_pred"] // 2 == pred_test["y_true"] // 2
sentimiento_ok = pred_test["y_pred"] % 2 == pred_test["y_true"] % 2
tabla_componentes = pd.DataFrame({
    "fusión final": [prenda_ok.mean(), sentimiento_ok.mean()],
    "rama sola": [accuracy_score(Y_PRENDA["test"], PRED_PRENDA["test"]),
                  accuracy_score(Y_SENTIMIENTO["test"][PARES["test"]], PRED_SENTIMIENTO["test"][PARES["test"]])],
}, index=["exactitud de la prenda", "exactitud del sentimiento"])
tipos_error = pd.Series({"correcto": (prenda_ok & sentimiento_ok).mean(), "solo prenda incorrecta": (~prenda_ok & sentimiento_ok).mean(),
                         "solo sentimiento incorrecto": (prenda_ok & ~sentimiento_ok).mean(), "ambos incorrectos": (~prenda_ok & ~sentimiento_ok).mean()},
                        name="proporción de pares")
display(tabla_componentes.round(4))

In [ ]:
# ============================================================
# Tipos de error en test
# ============================================================
display(tipos_error.to_frame().round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿La fusión conserva la exactitud de cada rama en su componente o pierde algo al combinarlas?
- Qué tipo de error domina y con qué rama se relaciona.

### 17.3 Métricas por clase conjunta

Se muestran las 20 clases conjuntas ordenadas por F1 en test, junto con el F1 de validación, para no sacar conclusiones de un solo conjunto.

`reporte_por_clase(y_true, y_pred)` retorna una tabla con precisión, exhaustividad, F1 y soporte por clase conjunta.

In [ ]:
# ============================================================
# Reporte por clase conjunta en test y validación
# ============================================================
def reporte_por_clase(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(20)), zero_division=0)
    return pd.DataFrame({"precision": p, "exhaustividad": r, "f1": f, "soporte": s}, index=CLASES_CONJUNTAS)


reporte = reporte_por_clase(pred_test["y_true"], pred_test["y_pred"])
reporte_val = reporte_por_clase(pred_val["y_true"], pred_val["y_pred"])
display(reporte.assign(f1_validacion=reporte_val["f1"]).sort_values("f1").round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Clases conjuntas con menor F1 y si coinciden con las prendas difíciles de la CNN.
- ¿Las clases negativas y positivas de una misma prenda rinden parecido?

### 17.4 Matriz de confusión

Matriz de 20 × 20 en test. Las confusiones entre clases de la misma prenda (sentimiento equivocado) quedan junto a la diagonal; las confusiones entre prendas forman bloques alejados de ella.

In [ ]:
# ============================================================
# Matriz de confusión conjunta en test
# ============================================================
matriz = confusion_matrix(pred_test["y_true"], pred_test["y_pred"], labels=list(range(20)))
fig = go.Figure(go.Heatmap(z=matriz, x=list(range(20)), y=list(range(20)), colorscale=[[0, "#FFFFFF"], [1, COLORES[0]]],
                           text=matriz, texttemplate="%{text}", textfont={"size": 9}))
fig.update_xaxes(tickvals=list(range(20)), ticktext=CLASES_CONJUNTAS, tickangle=-60, title_text="predicha")
fig.update_yaxes(tickvals=list(range(20)), ticktext=CLASES_CONJUNTAS, autorange="reversed", title_text="real")
fig.update_layout(title="Matriz de confusión del modelo final (test)", height=800)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Pares de clases conjuntas más confundidos y si la confusión es de prenda o de sentimiento.

## 18. Comparación entre la configuración base y la final

Hiperparámetros que cambiaron. Esta tabla compara configuraciones, no desempeño: el modelo final de una sola semilla es una estimación optimista.

In [ ]:
# ============================================================
# Configuración base frente a final
# ============================================================
config_base_vs_final = pd.DataFrame({"base": pd.Series(CONFIG_BASE).astype(str), "final": pd.Series(CONFIG_FINAL).astype(str)})
config_base_vs_final["cambia"] = config_base_vs_final["base"] != config_base_vs_final["final"]
display(config_base_vs_final)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Hiperparámetros que cambiaron y cuáles de esas decisiones superan MARGEN_ERROR.

## 19. Síntesis de los experimentos

Tabla con todas las variantes entrenadas o recuperadas, en el orden de los experimentos.

In [ ]:
# ============================================================
# Resumen de todas las variantes
# ============================================================
resumen_general = pd.concat(resultados_experimentos, ignore_index=True)
display(resumen_general[["experimento"] + COLUMNAS_TABLA].round(4))

## 20. Verificaciones de la dependencia del orden

Sin usar test, se prueban sobre la configuración final:

- **Tasa × lote:** ambos duplicados a la vez.
- **Capacidad × regularización:** el doble de unidades con 0,25 más de Dropout.
- **Técnica de fusión:** se eligió con la configuración base; se prueba la otra técnica con la configuración final.

Después se repite el modelo final con las mismas 5 semillas del base. Una diferencia mayor que `MARGEN_ERROR` indica que el orden de las decisiones condicionó el resultado.

In [ ]:
# ============================================================
# Verificaciones 20.1
# ============================================================
VERIFICACIONES = {
    "20.1a Tasa ×2 con lote ×2": dict(learning_rate=CONFIG_FINAL["learning_rate"] * 2, batch_size=CONFIG_FINAL["batch_size"] * 2),
    "20.1b Unidades ×2 con Dropout +0,25": dict(unidades=CONFIG_FINAL["unidades"] * 2, dropout=min(0.75, CONFIG_FINAL["dropout"] + 0.25)),
    "20.1c Otra técnica con la configuración final": dict(tecnica="concatenacion" if CONFIG_FINAL["tecnica"] == "atencion" else "atencion"),
}
filas_verificacion = [{"verificacion": "Configuración final", **fila_final}]
for nombre, cambios in VERIFICACIONES.items():
    fila, _, _ = entrenar_y_registrar({**CONFIG_FINAL, **cambios})
    print(f"{nombre}: {fila['origen']}")
    filas_verificacion.append({"verificacion": nombre, **fila})
tabla_verificacion = pd.DataFrame(filas_verificacion)
tabla_verificacion["diferencia_vs_final"] = tabla_verificacion["f1_macro_val"] - fila_final["f1_macro_val"]
tabla_verificacion["supera_margen"] = tabla_verificacion["diferencia_vs_final"].abs() > MARGEN_ERROR

In [ ]:
# ============================================================
# Resultados de las verificaciones 20.1
# ============================================================
display(tabla_verificacion[["verificacion", "f1_macro_val", "exactitud_val", "loss_val", "mejor_epoca", "parametros",
                            "diferencia_vs_final", "supera_margen", "origen"]].round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Alguna combinación supera a la configuración final por más que MARGEN_ERROR?
- Si es así, qué decisión quedó condicionada por el orden (si se adopta, se decide por criterio contextual).

### 20.2 Configuración final con 5 semillas

Se reentrena la configuración final con las mismas semillas del rango de error y se compara su F1 macro con el del modelo base.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración final
# ============================================================
filas_final_semillas = [{"semilla": SEED, **fila_final}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_FINAL, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_final_semillas.append({"semilla": semilla, **fila})
tabla_final_semillas = pd.DataFrame(filas_final_semillas)

In [ ]:
# ============================================================
# Comparación justa: base frente a final con 5 semillas
# ============================================================
comparacion_justa = pd.DataFrame({
    "base": tabla_variabilidad["f1_macro_val"].agg(["mean", "std", "min", "max"]),
    "final": tabla_final_semillas["f1_macro_val"].agg(["mean", "std", "min", "max"]),
})
comparacion_justa["diferencia"] = comparacion_justa["final"] - comparacion_justa["base"]
print(f"Mejora media: {comparacion_justa.loc['mean', 'diferencia'] * 100:.2f} pp | MARGEN_ERROR: {MARGEN_ERROR * 100:.2f} pp")
display(comparacion_justa.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Mejora media con 5 semillas frente a la estimación de una sola semilla (cota optimista).
- Advertencia: muchas variantes se evaluaron sobre la misma validación; el test es la estimación independiente.

## 21. Pesos de la atención entre modalidades

En el modelo de atención del experimento 12.1, cada *token* (imagen y texto) reparte su atención entre sí mismo y el otro. Se registran los pesos en los pares de validación y se promedian sobre los pares y las cabezas: una matriz de 2 × 2 en que cada fila es la modalidad que consulta y cada columna la modalidad consultada. Un peso alto fuera de la diagonal indica que una modalidad toma información de la otra antes de clasificar.

`pesos_modalidades(modelo, datos, config)` retorna esa matriz promedio.

In [ ]:
# ============================================================
# Atención promedio entre modalidades (validación)
# ============================================================
def pesos_modalidades(modelo, datos, config, lote=4096):
    modelo.eval()
    modelo.bloque.guardar_atencion = True
    suma = torch.zeros(2, 2, device=device)
    with torch.no_grad():
        for inicio in range(0, len(datos["y"]), lote):
            indices = torch.arange(inicio, min(inicio + lote, len(datos["y"])), device=device)
            modelo(*preparar(datos, indices, config))
            suma += modelo.bloque.atencion.mean(1).sum(0)
    modelo.bloque.guardar_atencion = False
    return (suma / len(datos["y"])).cpu().numpy()


config_atencion = {**historial_experimentos[1][1], **variantes_tecnica["atención entre modalidades"]}
matriz_atencion = pesos_modalidades(modelos_tecnica["atención entre modalidades"], DATOS["val"], config_atencion)
display(pd.DataFrame(matriz_atencion, index=["consulta: imagen", "consulta: texto"],
                     columns=["peso sobre imagen", "peso sobre texto"]).round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Cada modalidad atiende sobre todo a sí misma o toma información de la otra?
- Relación con la independencia entre prenda y sentimiento en estos pares.

## 22. Demostración del sistema completo

`predecir_par(imagen, texto)` recorre el sistema de punta a punta: la imagen pasa por la CNN congelada, el texto se limpia, se tokeniza y pasa por el modelo de texto congelado, ambas representaciones se estandarizan (si la configuración final lo indica) y el clasificador fusionado entrega las probabilidades de las 20 clases conjuntas. Retorna una tabla con las 3 clases más probables. Sirve para la ejecución en vivo: se puede cambiar el índice de la imagen o escribir otra reseña.

In [ ]:
# ============================================================
# Predicción de un par imagen–reseña
# ============================================================
def predecir_par(imagen, texto):
    ids = [STOI.get(t, UNK_IDX) for t in tokenizar(texto)[:MAXLEN]] or [UNK_IDX]
    with torch.no_grad():
        img = modelo_cnn.representacion(preparar_imagenes(imagen[None], CONFIG_CNN, MEDIA_PIXEL, STD_PIXEL))
        txt = modelo_texto.representacion(torch.tensor([ids], device=device), torch.tensor([len(ids)], device=device))
        img, txt = preparar({"img": img, "txt": txt}, torch.tensor([0], device=device), CONFIG_FINAL)
        modelo_final.eval()
        probas = torch.softmax(modelo_final(img, txt), 1)[0].cpu().numpy()
    mejores = np.argsort(-probas)[:3]
    return pd.DataFrame({"clase conjunta": [CLASES_CONJUNTAS[i] for i in mejores], "probabilidad": probas[mejores].round(4)})


INDICE_IMAGEN = 0
RESENA = "One of the best films I have seen this year, the story is moving and the cast is wonderful."
print("Prenda real:", CLASES[Y_PRENDA["test"][INDICE_IMAGEN]])
display(predecir_par(IMAGENES["test"][INDICE_IMAGEN], RESENA))

In [ ]:
# ============================================================
# Imagen del par de la demostración
# ============================================================
fig = px.imshow(IMAGENES["test"][INDICE_IMAGEN], color_continuous_scale="gray_r", height=320, width=320)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(title=f"Imagen de test {INDICE_IMAGEN}", coloraxis_showscale=False)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿La clase más probable combina correctamente la prenda de la imagen con el sentimiento de la reseña?

## 23. Conclusiones generales

*(completar después de ejecutar, con las cifras obtenidas: aporte de la fusión frente a cada modalidad, comparación entre técnicas, cercanía a la fusión tardía, efecto de los hiperparámetros, decisiones en empate y limitaciones del emparejamiento sintético)*

### 23.1 Mejoras propuestas

*(completar después de ejecutar)*

## 24. Conclusión final

*(completar después de ejecutar)*

## 25. Referencias

- Baltrušaitis, T., Ahuja, C. y Morency, L.-P. (2019). *Multimodal Machine Learning: A Survey and Taxonomy*. IEEE Transactions on Pattern Analysis and Machine Intelligence, 41(2), 423–443. arXiv:1705.09406.
- Kingma, D. P. y Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR. arXiv:1412.6980.
- Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y. y Potts, C. (2011). *Learning Word Vectors for Sentiment Analysis*. ACL.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. y Salakhutdinov, R. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR, 15, 1929–1958.
- Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J. y Wojna, Z. (2016). *Rethinking the Inception Architecture for Computer Vision*. CVPR. arXiv:1512.00567.
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł. y Polosukhin, I. (2017). *Attention Is All You Need*. NeurIPS. arXiv:1706.03762.
- Xiao, H., Rasul, K. y Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*. arXiv:1708.07747.
- PyTorch. *Documentación oficial*. https://pytorch.org/docs/stable/
- *EFT125_DLY0100_Estudiante* y *ET_Ejemplo_sentimiento_junto_clasificacion*. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).

## 26. Anexo A — Síntesis visual para la presentación

Gráficos en formato 16:9 con la paleta de la presentación, generados desde las variables ya calculadas (no entrenan). Cada uno se exporta a HTML en `respaldo_eft_fusion/graficos/`; el PNG se obtiene con el botón de cámara del gráfico o directamente si está instalado `kaleido`.

`estilo_presentacion(fig, titulo, subtitulo)` aplica el estilo común y retorna la figura. `exportar(fig, nombre)` guarda el gráfico en HTML (y en PNG si es posible).

In [ ]:
# ============================================================
# Estilo y exportación de los gráficos de la presentación
# ============================================================
RUTA_RESPALDO = "respaldo_eft_fusion"
RUTA_GRAFICOS = f"{RUTA_RESPALDO}/graficos"
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
AZUL_MARINO = "#0E2747"
ANCHO_PNG, ALTO_PNG = 1600, 900


def estilo_presentacion(fig, titulo, subtitulo=None):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:15px;color:#555555'>{subtitulo}</span>"
    fig.update_layout(template="plotly_white", title={"text": texto, "x": 0.02},
                      font={"family": "Arial", "size": 16, "color": AZUL_MARINO},
                      height=560, margin={"t": 110, "l": 80, "r": 40, "b": 80},
                      legend={"orientation": "h", "y": -0.18, "title": None})
    return fig


def exportar(fig, nombre):
    fig.write_html(f"{RUTA_GRAFICOS}/{nombre}.html", include_plotlyjs="cdn",
                   config={"toImageButtonOptions": {"format": "png", "width": ANCHO_PNG, "height": ALTO_PNG,
                                                    "scale": 2, "filename": nombre}})
    try:
        fig.write_image(f"{RUTA_GRAFICOS}/{nombre}.png", width=ANCHO_PNG, height=ALTO_PNG, scale=2)
        print(f"{nombre}: guardado en HTML y PNG")
    except Exception:
        print(f"{nombre}: guardado en HTML (PNG desde el botón de cámara del gráfico)")

### A.1 Aporte de la fusión en test

In [ ]:
# ============================================================
# A.1 Modalidades, fusión tardía y fusión final en test
# ============================================================
fig = go.Figure(go.Bar(x=tabla_test["modelo"], y=tabla_test["exactitud"],
                       marker_color=[COLORES[2]] * (len(tabla_test) - 1) + [COLORES[0]],
                       text=[f"{v:.3f}" for v in tabla_test["exactitud"]], textposition="outside"))
fig.add_hline(y=producto_test, line_dash="dash", line_color=AMARILLO, annotation_text="producto de las exactitudes de cada rama")
fig.update_yaxes(title_text="exactitud conjunta (test)", range=[0, 1.05])
estilo_presentacion(fig, "Ninguna modalidad sola resuelve la tarea conjunta", "Exactitud en los 10.000 pares de test")
exportar(fig, "a1_aporte_fusion")
fig.show()

### A.2 Técnicas de fusión

In [ ]:
# ============================================================
# A.2 Concatenación frente a atención: F1 de validación por época
# ============================================================
fig = go.Figure()
for i, (etiqueta, historia) in enumerate(historias_tecnica.items()):
    fig.add_trace(go.Scatter(x=list(range(1, len(historia["f1_val"]) + 1)), y=historia["f1_val"], name=etiqueta,
                             line={"color": [COLORES[0], AMARILLO][i % 2], "width": 3}))
fig.update_xaxes(title_text="época")
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Concatenación frente a atención entre modalidades", "F1 macro de validación por época")
exportar(fig, "a2_tecnicas")
fig.show()

### A.3 Tipos de error en test

In [ ]:
# ============================================================
# A.3 Descomposición de los errores
# ============================================================
fig = go.Figure(go.Bar(x=tipos_error.index, y=tipos_error.values * 100, marker_color=[COLORES[0], COLORES[1], COLORES[2], AMARILLO],
                       text=[f"{v * 100:.1f} %" for v in tipos_error.values], textposition="outside"))
fig.update_yaxes(title_text="% de los pares de test", range=[0, 105])
estilo_presentacion(fig, "Dónde se equivoca el sistema", "Pares de test según qué componente falla")
exportar(fig, "a3_tipos_error")
fig.show()

### A.4 Modelo base frente a modelo final (5 semillas)

In [ ]:
# ============================================================
# A.4 Base frente a final con 5 semillas
# ============================================================
fig = go.Figure()
for nombre, tabla, color in [("Base", tabla_variabilidad, COLORES[2]), ("Final", tabla_final_semillas, COLORES[0])]:
    fig.add_trace(go.Box(y=tabla["f1_macro_val"], name=nombre, marker_color=color, boxpoints="all", jitter=0.3))
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Configuración base frente a final", "F1 macro de validación con las mismas 5 semillas")
exportar(fig, "a4_base_vs_final")
fig.show()

### A.5 Impacto de cada decisión

In [ ]:
# ============================================================
# A.5 Aporte de cada decisión
# ============================================================
datos = tabla_decisiones.dropna(subset=["aporte"])
fig = go.Figure(go.Bar(x=datos["experimento"], y=datos["aporte"] * 100,
                       marker_color=[COLORES[0] if s else COLORES[2] for s in datos["supera_margen"]],
                       text=[f"{v * 100:+.2f}" for v in datos["aporte"]], textposition="outside"))
fig.add_hline(y=MARGEN_ERROR * 100, line_dash="dash", line_color=AMARILLO, annotation_text="rango de error")
fig.update_yaxes(title_text="aporte al F1 macro (pp)")
estilo_presentacion(fig, "Aporte de cada decisión", "Azul oscuro: supera el rango de error")
exportar(fig, "a5_decisiones")
fig.show()

## 27. Anexo B — Herramientas y tecnologías de apoyo utilizadas

| Categoría | Herramienta | Uso |
|---|---|---|
| Entorno de ejecución | Google Colab con GPU NVIDIA T4 | Extracción de representaciones y entrenamiento de los clasificadores fusionados |
| Editor | Visual Studio Code | Edición del notebook y de los scripts que lo generan; la ejecución se hizo en Colab, desde el navegador |
| Asistente de IA | Claude Opus 5.5 (Anthropic) | Estructuración del notebook, apoyo en el código y en la redacción; los resultados, decisiones e interpretaciones se revisaron con las salidas del propio notebook |
| Framework | PyTorch (https://pytorch.org) | Reconstrucción de los modelos de imagen y texto, y clasificadores fusionados |
| Librerías | NumPy (https://numpy.org), Pandas (https://pandas.pydata.org), scikit-learn (https://scikit-learn.org), Plotly (https://plotly.com/python/) | Arreglos, tablas, métricas y gráficos |
| Repositorio | GitHub | Datos, modelos entrenados, resultados guardados y versiones del notebook |

## 28. Respaldo de resultados

Se guardan en `respaldo_eft_fusion/` y se comprimen en `respaldo_eft_fusion.zip` las tablas, las historias, los pares de cada partición y el modelo final de fusión. Con un servidor remoto (Colab), los archivos quedan en el servidor. Con `DESCARGAR_RESULTADOS = True`, la celda descarga el `.zip` sola; si la descarga no llega (por ejemplo, desde VS Code), hay que bajarlo desde el panel *Archivos* antes de cerrar la sesión. El `.zip` ya incluye `resultados_fusion_guardados.json`.

In [ ]:
# ============================================================
# Respaldo de resultados
# ============================================================
os.makedirs(RUTA_RESPALDO, exist_ok=True)
resumen_general.to_csv(f"{RUTA_RESPALDO}/experimentos_todas_las_ejecuciones.csv", index=False)
tabla_variabilidad.to_csv(f"{RUTA_RESPALDO}/variabilidad.csv", index=False)
tabla_lineas_base_val.to_csv(f"{RUTA_RESPALDO}/lineas_base_validacion.csv", index=False)
usos.to_csv(f"{RUTA_RESPALDO}/emparejamiento.csv")
tabla_epocas.to_csv(f"{RUTA_RESPALDO}/epocas_sin_early_stopping.csv")
tabla_decisiones.to_csv(f"{RUTA_RESPALDO}/decisiones.csv", index=False)
config_base_vs_final.to_csv(f"{RUTA_RESPALDO}/config_base_vs_final.csv")
tabla_verificacion.to_csv(f"{RUTA_RESPALDO}/verificaciones_orden.csv", index=False)
tabla_final_semillas.to_csv(f"{RUTA_RESPALDO}/final_5_semillas.csv", index=False)
comparacion_justa.to_csv(f"{RUTA_RESPALDO}/comparacion_base_final.csv")
tabla_test.to_csv(f"{RUTA_RESPALDO}/metricas_test.csv", index=False)
tabla_componentes.to_csv(f"{RUTA_RESPALDO}/exactitud_por_componente.csv")
tipos_error.to_csv(f"{RUTA_RESPALDO}/tipos_de_error.csv")
reporte.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase_test.csv")
reporte_val.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase_validacion.csv")
pd.DataFrame(matriz, index=CLASES_CONJUNTAS, columns=CLASES_CONJUNTAS).to_csv(f"{RUTA_RESPALDO}/matriz_confusion.csv")
pd.DataFrame(matriz_atencion, index=["imagen", "texto"], columns=["imagen", "texto"]).to_csv(f"{RUTA_RESPALDO}/atencion_modalidades.csv")
with open(f"{RUTA_RESPALDO}/historias.json", "w") as f:
    json.dump({"base": historia_base, "final": historia_final, "sin_early_stopping": historia_epocas,
               "tecnica": historias_tecnica}, f)
np.save(f"{RUTA_RESPALDO}/y_test_proba.npy", pred_test["probas"])
torch.save({"config": CONFIG_FINAL, "estado": modelo_final.state_dict(), "clases": CLASES_CONJUNTAS,
            "estadisticos": {m: torch.stack(ESTADISTICOS[m]) for m in ESTADISTICOS},
            **{f"pares_{p}": torch.as_tensor(PARES[p]) for p in PARES}}, f"{RUTA_RESPALDO}/fusion_final.pt")
guardar_corridas()
shutil.copy(ARCHIVO_RESULTADOS, f"{RUTA_RESPALDO}/{ARCHIVO_RESULTADOS}")
shutil.make_archive(RUTA_RESPALDO, "zip", RUTA_RESPALDO)
print(f"Respaldo guardado en {RUTA_RESPALDO}.zip ({os.path.getsize(RUTA_RESPALDO + '.zip') / 1e6:.1f} MB)")
if DESCARGAR_RESULTADOS:
    try:
        files.download(f"{RUTA_RESPALDO}.zip")
    except Exception as error:
        print(f"Descarga automática no disponible ({type(error).__name__}): descargar {RUTA_RESPALDO}.zip desde el panel Archivos")

## 29. Descripción de las salidas

| Archivo | Contenido | Uso posterior |
|---|---|---|
| `resultados_fusion_guardados.json` | Métricas de cada corrida, por clave de configuración y semilla | Subirlo a la raíz del repositorio para que la siguiente ejecución no repita corridas |
| `respaldo_eft_fusion/fusion_final.pt` | Configuración, pesos, clases conjuntas, estadísticos de estandarización y pares de cada partición del modelo final | Reproducir la demostración sin reentrenar |
| `respaldo_eft_fusion/*.csv` | Tablas de experimentos, variabilidad, emparejamiento, decisiones, verificaciones, test, componentes, tipos de error, matriz de confusión y atención entre modalidades | Regenerar tablas y gráficos sin reentrenar |
| `respaldo_eft_fusion/historias.json` | Curvas por época del base, del final, sin Early Stopping y de ambas técnicas | Gráficos de la presentación |
| `respaldo_eft_fusion/y_test_proba.npy` | Probabilidades del modelo final en los pares de test | Análisis de errores |
| `respaldo_eft_fusion/graficos/` | Gráficos del anexo A en HTML (y PNG si hay `kaleido`) | Diapositivas de la presentación |
| `respaldo_eft_fusion.zip` | Todo lo anterior comprimido | Descarga desde Colab |

## 30. Cierre del entorno de ejecución

Al terminar, el entorno de Colab se libera con `runtime.unassign()`, para no mantener la GPU asignada sin uso. Colab no publica cuánto tiempo de inactividad tolera antes de cerrar una sesión (las guías lo estiman en unos 90 minutos), y limita más a las cuentas con uso intensivo reciente.

- **Por qué con temporizador:** en las ejecuciones de este proyecto, las descargas de `files.download` llegaron al navegador recién cuando terminó la última celda. Si el cierre se ejecutara en esta misma celda, cortaría esas descargas. Por eso, un temporizador (`threading.Timer`) programa el cierre `ESPERA_CIERRE` segundos después, y la celda termina de inmediato: el entorno queda libre para completar las descargas y luego se cierra.
- **Control manual:** `temporizador_cierre.cancel()` cancela el cierre si hace falta seguir trabajando; con `CERRAR_ENTORNO = False` no se programa.
- **Fuera de Colab** (por ejemplo, en VS Code con kernel de Colab), el cierre puede no estar disponible: se imprime un aviso y el entorno se cierra a mano (*Entorno de ejecución → Desconectar y eliminar el entorno de ejecución*).

In [ ]:
# ============================================================
# Cierre del entorno de ejecución
# ============================================================
CERRAR_ENTORNO = True
ESPERA_CIERRE = 120


def cerrar_entorno():
    try:
        runtime.unassign()
    except Exception as error:
        print(f"No se pudo cerrar el entorno ({type(error).__name__}): cerrarlo desde Entorno de ejecución → Desconectar y eliminar")


temporizador_cierre = threading.Timer(ESPERA_CIERRE, cerrar_entorno)
if CERRAR_ENTORNO and runtime is not None:
    temporizador_cierre.start()
    print(f"El entorno se liberará en {ESPERA_CIERRE} s, después de las descargas pendientes (cancelar: temporizador_cierre.cancel())")
else:
    print("Cierre automático desactivado o fuera de Colab: cerrar el entorno a mano")